# HTBoost v5 — Per-Security, Horizon-Sweep, Train+OOS, Block-CV Robustness

**This notebook extends v4. v4 is left intact.** Same inviolable discipline:
per-security single-target HTBoost on **Norwegian** swaps; target `y = level.shift(-h) − level`;
a *null-confirmation* harness — strong in-sample fit collapsing to OOS ≈ 0.50 is the
**designed, expected** outcome, and the in-sample (Ch.7) vs OOS (Ch.8) gap **is the finding**.

**What v5 adds (all a-priori, none tuned against OOS):**
1. **Less overfitting.** The 178 cross-market `xm_*` columns are compressed by **PCA fit on
   training rows only** (round-tripped through every fold). L2 (`lambda`) is pre-committed by an
   **inner-CV grid on the training window**; `nfold` raised above 1 for genuine internal CV.
2. **Documented feature engineering.** Norway block confirmed wired + audited; a **feature
   taxonomy** tags every column to a bucket (`curve/momentum/vol/macro/credit/cross_market/norway/carry_roll`).
3. **Train AND OOS captured for every run.** One row per `(security, tenor, horizon, fold, regime, sample)`
   with the full metric set: DirAcc, R², Campbell–Thompson OOS-R², Clark–West, Diebold–Mariano–Harvey
   (HAC/Newey–West, lag≈H−1), Pesaran–Timmermann, one-sided binomial. MTC across the whole grid.
4. **Pooled-comparable schema.** A single `compute_metrics_row` and one tidy long CSV
   (`v5_metrics_long.csv`); a comparison cell merges a pooled CSV if present, skips gracefully if not.
5. **Honest horizon sweep.** `H ∈ {1,5,21,63}` (+`126/252` gated on data length); hyperparameters
   selected **once, on train only, before the sweep**, then **frozen** for every horizon and tenor.
6. **Block-CV / leave-one-regime-out robustness.** A *second*, deliberately **non-causal**
   validation scheme (trains on data surrounding the held-out block, future included) with **two-sided
   purge+embargo**. It is a **learnability / regime-transfer diagnostic, NOT a forecast** and never
   enters the trading sim.

**Decision rule (stated before running).** v5 confirms the null if the per-security OOS DirAcc
distribution is noise-consistent and no security survives multiple-testing correction.
**A genuine lead requires ALL of:** (1) clearing Bonferroni- or BH-FDR-corrected significance
(N = grid size, recomputed); (2) positive DirAcc across **multiple** regimes (GFC, ZIRP, COVID, Hiking),
not just Hiking. One-regime strength is not a lead — the standard that ruled out `NOR_30Y`.

**Block-CV interpretation rule (ex ante, asymmetric).**
- block-CV OOS ≈ 0.50 → **stronger** confirmation of the null (even with future data, nothing transfers).
- block-CV OOS > 0.50 while walk-forward ≈ 0.50 → attributable to the **future-information advantage /
  larger training set**; it flags **non-stationarity**, it is **NOT tradeable skill**.
- Cite Bergmeir, Hyndman & Koo (2018) and López de Prado (2018, purged K-fold + embargo).

## Setup
Import the scientific-Python stack, the statistical-test libraries, and the Julia bridge (`juliacall`) used to call HTBoost, plus the project's `data_loader` helper.

In [ ]:
import re
import os
import sys
import json
import hashlib
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest, binom, norm
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from juliacall import Main as jl

# Put the repo root (dir containing src/) on sys.path so the flat root-level module(s)
# (thesis_style) import whether cwd is the repo root or notebooks/.
_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)
from src.data.bloomberg import load_data  # was: data_loader shim (broke from notebooks/ cwd)
import src.config as config  # shared constants (JULIA_SEED, XM_PCA_*)
# Shared thesis figure style (Okabe-Ito palette, vector PDF, despined axes, regime colours).
from thesis_style import apply_thesis_style, OKABE_ITO, REGIME_COLORS, SERIES_COLOR
apply_thesis_style()

print('Imports OK (v5: + PCA/StandardScaler, statsmodels HAC, scipy.norm, hashlib)')

### Julia runtime
Load the Julia packages HTBoost needs and remove any leftover worker processes so each fit runs single-process, then pin the RNG seed for reproducibility.

In [ ]:
jl.seval('using DataFrames')
jl.seval('using HybridTreeBoosting')
jl.seval('using Distributed')
jl.seval('using SharedArrays')
jl.seval('using Dates')
jl.seval(
    'ws = workers(); '
    'if length(ws) > 0 && ws != [1]; rmprocs(ws); end'
)
# Pin the Julia RNG for reproducibility (same seed as the pooled v5 notebook).
jl.seval(f'using Random; Random.seed!({config.JULIA_SEED})')
print(f'Julia procs after cleanup: {jl.nprocs()}  (should be 1)')

### Configuration
All run parameters in one place: horizons, the regularization grid, the walk-forward and block-CV fold dates, the PCA settings, minimum-observation thresholds, and the switches that gate the heavy sweeps. None of these are chosen from out-of-sample results.

In [ ]:
# ── CONFIG — all run parameters live here ────────────────────────────────────────────────
# Horizon grid, PCA compression, the pre-committed inner-CV grid, block-CV folds +
# embargo, and the run switches.
# NOTHING here is chosen by looking at an OOS/regime fold.

# ── Primary horizon + the pre-committed horizon GRID ───────────────────
H                = 5            # default/primary horizon (smoke, leakage audit, Part C default)
H_GRID           = [1, 5, 21, 63]          # 1d, 1w, 1m, 3m — always run
H_GRID_LONG      = [126, 252]              # 6m, 12m — only if data length supports the folds
OVERLAP          = H - 1                    # label-overlap purge; ALWAYS H-1 per horizon (invariant)

LOSS             = 't'
MODALITY         = 'accurate'  # strongest HTBoost setting (committed scientific config)
UNIVERSE_MIN_OBS = 500
MIN_TRAIN_OBS    = 252         # ≥1 year training per fold
MIN_TEST_OBS     = 20          # ≥20 OOS obs to score a fold
ALPHA_MT         = 0.05        # multiple-testing alpha
SMOKE_SEC        = 'NOR_10Y'   # NOR pilot (this study restricts to Norwegian swaps)
TARGET_LAGS      = 5           # AR lags of chg_1d; fixed, identical for all securities/horizons

OUT_DIR = f'{_ROOT}/results/v5_nor'
os.makedirs(OUT_DIR, exist_ok=True)

# ── PCA compression of the cross-market xm_* block (fit on TRAIN only) ───
# The xm_* block (~178 cols for NOR_10Y) is the named overfitting surface. We
# replace it with k principal components, fit on the fold's TRAINING rows only,
# applied identically to every tenor/horizon/fold. Pre-committed rule:
#   k = smallest #components explaining ≥ XM_PCA_VAR of TRAIN variance, capped at XM_PCA_KMAX.
# Standardisation uses TRAIN mean/std. This lives in the FIT PATH, never in
# _features_for_security_v4 (the audit cannot validate a fold-dependent transform).
XM_PCA_ENABLE    = True
XM_PCA_VAR       = 0.95        # explained-variance target (a-priori)
XM_PCA_KMAX      = 12          # hard cap on #components (a-priori)

# ── L2 regularization fixed A PRIORI (no data-driven lambda tuning) ──
# For a null-confirmation harness we do NOT tune lambda. It is pre-committed to the
# MIDPOINT of LAM_GRID and applied uniformly to every horizon/tenor/fold/scheme;
# robustness across lambda is REPORTED by the λ-sensitivity cell (never a selection).
# depth / ntrees are selected by modality='accurate's internal CV per fit (a-priori
# uniform rule); gridding depth would double-CV it (documented HTBoost behaviour).
LAM_GRID         = [0.05, 0.10, 0.20]   # lambda grid: midpoint pre-committed; rest for the sensitivity report
NFOLD_INTERNAL   = 4                    # raised above 1 → genuine internal block-CV early stopping
# FROZEN_CONFIG is set in the regularization cell (lambda = LAM_GRID midpoint) and reused EVERYWHERE.
LAM              = 0.10                  # provisional; set to the a-priori midpoint in the regularization cell
FROZEN_CONFIG    = None                 # set by the regularization cell; dict(lam, depth, nfold, modality, loss)

# ── block-CV / leave-one-regime-out folds + embargo ─────────────────────
# Full sample (2007+), aligned to regimes. EMBARGO scales with H.
BLOCK_CV_FOLDS = [
    ('Block_GFC',    '2007-01-01', '2012-12-31', 'GFC'),
    ('Block_ZIRP',   '2013-01-01', '2019-12-31', 'ZIRP'),
    ('Block_COVID',  '2020-01-01', '2021-12-31', 'COVID'),
    ('Block_Hiking', '2022-01-01', '2026-12-31', 'Hiking'),
]
def EMBARGO_FOR_H(h):
    """López de Prado embargo (business days); scales with H. Two-sided in block-CV."""
    return max(int(h), 10)

# ── Run switches (gate the heavy sweeps) ─────────────────────────────────────
RUN_WALK_FORWARD = True
RUN_BLOCK_CV     = True
RUN_LORO         = True
# QUICK_MODE is a DEVELOPMENT/EXECUTION-BUDGET switch only — it does NOT change any
# model-selection decision (those stay frozen). It shrinks modality / grid / subset
# so the whole notebook runs end-to-end quickly to verify wiring. Default OFF: the
# thesis run uses the committed config. When ON, results are NOT for reporting.
QUICK_MODE       = False
QUICK_MODALITY   = 'fast'
QUICK_TENORS     = ['NOR_10Y']
QUICK_HGRID      = [5]

# ── pooled-notebook comparison ──────────────────────────────────────────
POOLED_CSV_PATH  = f'{OUT_DIR}/pooled_metrics_long.csv'   # configurable; absent ⇒ comparison skips

# ── Norway external-data fetch (Part B) ───────────────────────────────────────
NOR_FETCH        = os.environ.get('NORWAY_LIVE_FETCH', '') == '1'   # cache-first default; set NORWAY_LIVE_FETCH=1 to refresh from live APIs
NOR_CACHE        = f'{_ROOT}/data/cache/norway_raw_features.csv'     # git-shipped cache, cwd-independent; a live refresh overwrites it
NOR_RELEASE_LAG_M = 1

# ── Walk-forward folds: expanding window, test windows map to regimes ─────────
WF_FOLDS = [
    # (name,          test_start,    test_end,      regime)
    ('GFC',           '2010-01-01',  '2012-12-31',  'GFC'),
    ('ZIRP_early',    '2013-01-01',  '2016-12-31',  'ZIRP'),
    ('ZIRP_late',     '2017-01-01',  '2019-12-31',  'ZIRP'),
    ('COVID',         '2020-01-01',  '2021-12-31',  'COVID'),
    ('Hiking',        '2022-01-01',  '2026-12-31',  'Hiking'),
]
FOLD_NAMES = [f[0] for f in WF_FOLDS]
NOTEBOOK_TAG = 'v5'
RUN_TS = pd.Timestamp.now(tz='UTC').isoformat()

# Seed the cache from the existing v4 cache so the Norway fetch can run offline
# rather than dropping NOR features when the live sources are unreachable.
if not os.path.exists(NOR_CACHE) and os.path.exists('htboost_results_v4_nor/norway_raw_features.csv'):
    import shutil; shutil.copy('htboost_results_v4_nor/norway_raw_features.csv', NOR_CACHE)

print(f'H(primary)={H}  H_GRID={H_GRID}  (+long {H_GRID_LONG} if data supports)')
print(f'LOSS={LOSS!r} MODALITY={MODALITY!r}  NFOLD_INTERNAL={NFOLD_INTERNAL}  TARGET_LAGS={TARGET_LAGS}')
print(f'PCA(xm_*): enable={XM_PCA_ENABLE} var≥{XM_PCA_VAR} kmax={XM_PCA_KMAX}')
print(f'inner-CV LAM grid: {LAM_GRID}  (depth/ntrees via modality internal CV)')
print(f'switches: WF={RUN_WALK_FORWARD} BLOCK_CV={RUN_BLOCK_CV} LORO={RUN_LORO}  QUICK_MODE={QUICK_MODE}')
print(f'OUT_DIR={OUT_DIR!r}')
print(f'Walk-forward folds ({len(WF_FOLDS)}):')
for f in WF_FOLDS:
    print(f'  {f[0]:<14s}  test [{f[1]} → {f[2]}]  regime={f[3]}')
print(f'Block-CV folds ({len(BLOCK_CV_FOLDS)}):')
for f in BLOCK_CV_FOLDS:
    print(f'  {f[0]:<14s}  block [{f[1]} → {f[2]}]  regime={f[3]}')

### Load data and define the universe
Load the swap panel, identify the swap columns, guard against duration columns leaking in, and restrict the study universe to Norwegian swaps with enough history.

In [ ]:
df_raw = load_data()

_SWAP_PAT = re.compile(r'^[A-Z]+_\d+[WMY]$')
SWAP_COLS  = sorted([c for c in df_raw.columns if _SWAP_PAT.match(c)])

# Duration.xlsx is NOT loaded by load_data() — guard anyway
dur_cols = [c for c in df_raw.columns if 'uration' in c.lower() or 'DV01' in c]
assert len(dur_cols) == 0, f'Duration columns found — exclude: {dur_cols}'

print(f'df_raw shape:    {df_raw.shape}')
print(f'Date range:      {df_raw.index.min().date()} → {df_raw.index.max().date()}')
print(f'Swap columns:    {len(SWAP_COLS)}')
print(f'Non-swap columns:{len(df_raw.columns) - len(SWAP_COLS)}')
print(f'Duration guard:  OK (0 duration columns in df_raw)')

# All available (non-swap) column names — inventory of what we can use as features
non_swap = [c for c in df_raw.columns if not _SWAP_PAT.match(c)]
print(f'\nNon-swap columns available for features ({len(non_swap)}):')
for c in non_swap:
    n = df_raw[c].notna().sum()
    print(f'  {c:<40s}  {n:5d} obs')

# Universe: freeze on full history (all available obs)
obs      = df_raw[SWAP_COLS].notna().sum()
UNIVERSE = sorted(obs[obs >= UNIVERSE_MIN_OBS].index.tolist())
print(f'\nUniverse (≥{UNIVERSE_MIN_OBS} obs): {len(UNIVERSE)} securities')


# ── PART A: restrict universe to Norwegian swaps ─────────────────────────────
UNIVERSE = [s for s in UNIVERSE if s.rsplit('_', 1)[0] == 'NOR']
print(f'\nPART A — Norwegian-swap universe ({len(UNIVERSE)} securities):')
for s in UNIVERSE:
    print(f'  {s:<10s}  {df_raw[s].notna().sum():5d} obs')
assert SMOKE_SEC in UNIVERSE, f'{SMOKE_SEC} not in NOR universe'

### Part B — Norway data fetch
Probe each public Norwegian/European data source live (Norges Bank, SSB, ECB, Riksbank), log what connects, and derive leakage-safe `nor_*` feature columns. Sources that fail are skipped, with the existing columns as the fallback.

In [ ]:
# ── PART B: Norway-specific data fetch + connectivity report ─────────────────
# Probe every source live. Integrate a series ONLY if the fetch returns usable data.
# Anything that fails is LOGGED and skipped (existing columns are the fallback).
#   Norges Bank (no auth):  EXR (EUR/NOK, USD/NOK, I-44), IR (policy rate),
#                           SHORT_RATES (NOWA), GOVT_GENERIC_RATES (govt yields)
#   SSB PxWebApi (no auth):  KPI YoY (03013), KPI-JAE YoY (05327), LFS unemp (13760)
#   ECB SDMX:                deposit facility rate;   Riksbank SWEA: policy rate
#   Brent / Brent-WTI / EIA gas:  SKIPPED (needs an API key) -> fall back to the
#                           existing oil_mom_* / natgas_mom_1m columns (already shift-1).
import io, urllib.request

NB_API  = 'https://data.norges-bank.no/api/data'
SSB_API = 'https://data.ssb.no/api/v0/en/table'
_HDRS   = {'User-Agent': 'master-thesis-research'}


def _http_get(url):
    return urllib.request.urlopen(urllib.request.Request(url, headers=_HDRS),
                                  timeout=30).read().decode('utf-8', 'replace')


def _http_post(url, payload):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={**_HDRS, 'Content-Type': 'application/json'})
    return urllib.request.urlopen(req, timeout=30).read().decode('utf-8', 'replace')


def _nb_series(flow, key, start, end):
    """Norges Bank SDMX -> pd.Series(OBS_VALUE) indexed by date."""
    url = f'{NB_API}/{flow}/{key}?format=csv&startPeriod={start}&endPeriod={end}'
    d = pd.read_csv(io.StringIO(_http_get(url)), sep=';')
    s = pd.Series(pd.to_numeric(d['OBS_VALUE'], errors='coerce').values,
                  index=pd.to_datetime(d['TIME_PERIOD'])).dropna().sort_index()
    return s[~s.index.duplicated(keep='last')]


def _ssb_monthly(table, content, filters, start, end):
    """SSB PxWebApi monthly -> pd.Series indexed by reference-month END timestamp."""
    query = [{'code': c, 'selection': {'filter': 'item', 'values': v}} for c, v in filters]
    query.append({'code': 'ContentsCode', 'selection': {'filter': 'item', 'values': [content]}})
    js  = json.loads(_http_post(f'{SSB_API}/{table}',
                                {'query': query, 'response': {'format': 'json-stat2'}}))
    idx, vals = js['dimension']['Tid']['category']['index'], js['value']
    out = {}
    for m, pos in sorted(idx.items(), key=lambda kv: kv[1]):
        if vals[pos] is None:
            continue
        out[pd.Timestamp(int(m[:4]), int(m[5:7]), 1) + pd.offsets.MonthEnd(0)] = float(vals[pos])
    s = pd.Series(out).sort_index()
    return s[(s.index >= start) & (s.index <= end)]


def _ecb_dfr(start):
    url = ('https://data-api.ecb.europa.eu/service/data/FM/'
           f'B.U2.EUR.4F.KR.DFR.LEV?format=csvdata&startPeriod={start}')
    d = pd.read_csv(io.StringIO(_http_get(url)))
    return pd.Series(pd.to_numeric(d['OBS_VALUE'], errors='coerce').values,
                     index=pd.to_datetime(d['TIME_PERIOD'])).dropna().sort_index()


def _riksbank(start):
    j = json.loads(_http_get(f'https://api.riksbank.se/swea/v1/Observations/SECBREPOEFF/{start}'))
    return pd.Series({pd.Timestamp(o['date']): float(o['value']) for o in j}).sort_index()


def fetch_norway_data(start, end):
    """Probe every Norway source. Returns (raw_df, report). Each source is wrapped:
    on failure it is logged and skipped, never fabricated."""
    a, b   = pd.Timestamp(start).strftime('%Y-%m-%d'), pd.Timestamp(end).strftime('%Y-%m-%d')
    raw, report = {}, []

    def _try(label, fn, freq):
        try:
            s = fn()
            d = s.dropna() if s is not None else pd.Series(dtype=float)
            if len(d) == 0:
                report.append((label, 'SKIP', 'empty', freq, 0, None, None)); return None
            report.append((label, 'OK', '', freq, len(d), d.index.min().date(), d.index.max().date()))
            return s
        except Exception as e:
            report.append((label, 'FAIL', repr(e)[:60], freq, 0, None, None)); return None

    raw['nb_eurnok'] = _try('NB EUR/NOK',  lambda: _nb_series('EXR', 'B.EUR.NOK.SP', a, b), 'daily')
    raw['nb_usdnok'] = _try('NB USD/NOK',  lambda: _nb_series('EXR', 'B.USD.NOK.SP', a, b), 'daily')
    raw['nb_i44']    = _try('NB I-44',     lambda: _nb_series('EXR', 'B.I44.NOK.SP', a, b), 'daily')
    raw['nb_polrate']= _try('NB policy rate', lambda: _nb_series('IR', 'B.KPRA.SD.R', a, b), 'daily')
    raw['nb_nowa']   = _try('NB NOWA',     lambda: _nb_series('SHORT_RATES', 'B.NOWA.ON.R', a, b), 'daily')
    for ten in ['3Y', '5Y', '10Y']:
        raw[f'nb_govt_{ten.lower()}'] = _try(
            f'NB govt {ten}', lambda t=ten: _nb_series('GOVT_GENERIC_RATES', f'B.{t}.GBON', a, b), 'daily')
    raw['ssb_kpi_yoy']   = _try('SSB KPI YoY', lambda: _ssb_monthly(
        '03013', 'Tolvmanedersendring', [('Konsumgrp', ['TOTAL'])], a, b), 'monthly')
    raw['ssb_kpijae_yoy']= _try('SSB KPI-JAE YoY', lambda: _ssb_monthly(
        '05327', 'Tolvmanedersendring', [('Konsumgrp', ['JAE_TOTAL'])], a, b), 'monthly')
    raw['ssb_unemp']     = _try('SSB LFS unemp', lambda: _ssb_monthly(
        '13760', 'Arbeidsledige',
        [('Kjonn', ['0']), ('Alder', ['15-74']), ('Justering', ['S'])], a, b), 'monthly')
    raw['ecb_dfr']    = _try('ECB deposit rate', lambda: _ecb_dfr(a), 'event')
    raw['rb_polrate'] = _try('Riksbank policy rate', lambda: _riksbank(a), 'daily')

    raw = {k: v for k, v in raw.items() if v is not None}
    return (pd.DataFrame(raw).sort_index() if raw else pd.DataFrame()), report


def _monthly_released_daily(s_refmonth, target_index, lag_m=NOR_RELEASE_LAG_M):
    """Reference-month value -> stamped at end of month M+lag (>= real SSB release),
    then ffill onto daily target_index. Conservative: never look-ahead."""
    if s_refmonth is None or len(s_refmonth.dropna()) == 0:
        return pd.Series(np.nan, index=target_index)
    sh = s_refmonth.dropna().sort_index().copy()
    sh.index = sh.index + pd.offsets.MonthEnd(lag_m)
    sh = sh[~sh.index.duplicated(keep='last')]
    return sh.reindex(target_index, method='ffill')


def add_norway_features_v4(df, nor_raw=None):
    """Leakage-safe NOR-gated feature columns (nor_*) from raw fetched series.
    Timing policy:
      FX (EUR/NOK, USD/NOK, I-44) ........ shift(1)  [USD-cross & index, prior close]
      policy rate / NOWA / govt yields ... contemporaneous  [NOK official same-session]
      ECB / Riksbank policy rates ........ contemporaneous  [administered step, ffill]
      monthly macro (KPI, KPI-JAE, unemp) release-aligned ffill + shift(1)
    Daily series read from df (joined); monthly series from nor_raw (native month index)."""
    df  = df.copy()
    idx = df.index

    def _raw(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=idx)

    # FX — shift(1)
    for srccol, lbl in [('nb_eurnok', 'eurnok'), ('nb_usdnok', 'usdnok'), ('nb_i44', 'i44')]:
        fx = _raw(srccol)
        if fx.notna().sum() == 0:
            continue
        df[f'nor_{lbl}_level']    = fx.shift(1)
        df[f'nor_{lbl}_chg_1d']   = fx.diff().shift(1)
        df[f'nor_{lbl}_mom_1m']   = fx.pct_change().rolling(21, min_periods=10).sum().shift(1)
        df[f'nor_{lbl}_mom_3m']   = fx.pct_change().rolling(63, min_periods=30).sum().shift(1)
        df[f'nor_{lbl}_rvol_20d'] = fx.pct_change().rolling(20, min_periods=5).std().shift(1)

    # Policy rate (styringsrenten) — contemporaneous; days-since-last-change
    pol = _raw('nb_polrate')
    if pol.notna().sum() > 0:
        df['nor_polrate_level']  = pol
        df['nor_polrate_chg_1d'] = pol.diff()
        grp = (pol.diff().fillna(0) != 0).cumsum()
        df['nor_polrate_days_since_chg'] = grp.groupby(grp).cumcount().astype(float)

    # NOWA + NIBOR-NOWA spread — contemporaneous (NOK same-session)
    nowa = _raw('nb_nowa')
    if nowa.notna().sum() > 0:
        df['nor_nowa_level']        = nowa
        df['nor_nibor_nowa_spread'] = _raw('NIBOR3M Index  (L1)') - nowa

    # Government bond yields — contemporaneous (NOK official)
    for ten in ['3y', '5y', '10y']:
        g = _raw(f'nb_govt_{ten}')
        if g.notna().sum() > 0:
            df[f'nor_govt_{ten}'] = g

    # ECB deposit rate / Riksbank policy rate — contemporaneous (administered step)
    ecb = _raw('ecb_dfr').reindex(idx).ffill()
    if ecb.notna().sum() > 0:
        df['nor_ecb_dfr_level'] = ecb
        df['nor_ecb_dfr_chg']   = ecb.diff()
    rb = _raw('rb_polrate').reindex(idx).ffill()
    if rb.notna().sum() > 0:
        df['nor_rb_rate_level'] = rb
        df['nor_rb_rate_chg']   = rb.diff()

    # Monthly macro — release-aligned ffill + shift(1)
    for srccol, dst in [('ssb_kpi_yoy', 'nor_kpi_yoy'),
                        ('ssb_kpijae_yoy', 'nor_kpijae_yoy'),
                        ('ssb_unemp', 'nor_unemp')]:
        mser = nor_raw[srccol] if (nor_raw is not None and srccol in nor_raw.columns) else None
        if mser is not None and mser.dropna().shape[0] > 0:
            df[dst] = _monthly_released_daily(mser, idx).shift(1)
    return df


# ── Execute fetch (live, with cache fallback) ────────────────────────────────
nor_raw = pd.DataFrame()
nor_report = []
if NOR_FETCH:
    try:
        nor_raw, nor_report = fetch_norway_data(df_raw.index.min(), df_raw.index.max())
        if not nor_raw.empty:
            nor_raw.to_csv(NOR_CACHE)
    except Exception as e:
        print(f'[WARN] live fetch failed ({repr(e)[:80]}); trying cache')
if nor_raw.empty and os.path.exists(NOR_CACHE):
    nor_raw = pd.read_csv(NOR_CACHE, index_col=0, parse_dates=True)
    nor_report = nor_report or [('cache load', 'OK', NOR_CACHE, 'mixed', len(nor_raw), None, None)]

print('=== PART B — connectivity report ===')
print(f'  {"source":24s} {"stat":4s} {"freq":7s} {"n_obs":>6s}  coverage')
print('  ' + '-' * 70)
for label, status, note, freq, n, lo, hi in nor_report:
    cov = f'{lo}..{hi}' if lo else ''
    print(f'  {label:24s} {status:4s} {freq:7s} {n:6d}  {cov}  {note}')
print('  SKIP Brent / Brent-WTI / EIA gas  (needs API key) -> fall back to oil_mom_*/natgas_mom_1m')

# Merge daily raw series into df_raw (monthly SSB series stay in nor_raw), then derive nor_*
_daily_raw = [c for c in nor_raw.columns if not c.startswith('ssb_')]
# Idempotent: only join raw daily series not already present (re-joining existing
# columns would raise an overlap error), then (re)derive nor_* from source columns.
_join_cols = [c for c in _daily_raw if c not in df_raw.columns]
if _join_cols:
    df_raw = df_raw.join(nor_raw[_join_cols], how='left')
df_raw = add_norway_features_v4(df_raw, nor_raw)
NOR_FEATURE_COLS = sorted(c for c in df_raw.columns if c.startswith('nor_'))
print(f'\n{len(NOR_FEATURE_COLS)} nor_* feature columns derived (gated to country==\'NOR\'):')
for c in NOR_FEATURE_COLS:
    print(f'  {c:30s} {df_raw[c].notna().sum():6d} obs')

### Macro/global feature builder
Define the timing-alignment policy and `add_macro_features_v4`, which builds the global macro, volatility, equity, commodity, credit and IBOR features. The policy lag keeps every exogenous feature observable at decision time.

In [ ]:
# ── TIMING ALIGNMENT POLICY ──────────────────────────────────────
# US-close exogenous series (VIX/MOVE/equity/commodity) are shifted by 1 day so a
# feature is always observable at decision time. Same-day US-close values are safe
# for US securities (SOFR) where feature and fixing are both US-close, but for
# non-US fixings (EUR, NOR fix intraday before NY close) the same-day US close is
# not yet available. Shifting US-close series by 1 day — feature[t] = series[t-1] —
# makes every feature observable at any market.
#
# Series NOT shifted (contemporaneous OK):
#   - own-security swap rate: target and feature are the same daily close
#   - country IBOR rates: same-country same-session fix
#   - monthly macro: ffill+shift(1)
#
# Series shifted by 1:
#   - VIX, MOVE, V2X, VXN, RVX, OVX, GVZ
#   - SPX, SX5E, MXWO, NDX
#   - Oil, copper, OPEC, natgas
#   - Breakeven inflation (TIPS), IG/HY credit spreads
#   - Additional: CPURNSA, PCE CORE
#
# Cross-market swap rates: shift(1) applied in _features_for_security_v4.

MATURITY_YEARS = {
    '1W': 1/52, '1M': 1/12, '3M': 0.25, '6M': 0.5,
    '1Y': 1.0,  '5Y': 5.0,  '10Y': 10.0, '15Y': 15.0, '30Y': 30.0,
}
COUNTRY_PRIMARY_IBOR = {
    'NOR':   ('NIBOR3M Index  (L1)',  'NIBOR6M Index'),
    'SWE':   ('STIB3M Index  (L1)',   None),
    'EUR':   ('EUR006M Index  (L1)',  'EUR003M Index'),
    'SOFR':  ('SOFRRATE Index  (L1)', None),
    'CAN':   ('CAONREPO Index  (L1)', None),
    'AUS':   ('BBSW3M Index  (L1)',   None),
    'POL':   ('WIBR3M Index  (L1)',   None),
    'BRAZ':  ('BZDIOVRA Index  (R2)', None),
    'CHIN':  ('CNRR007 Index  (R2)',  None),
    'TURK':  ('MUTKCALM Index  (R1)', None),
    'SONIA': ('SONIO/N Index  (L1)',  None),
}


def add_macro_features_v4(df):
    df = df.copy()

    def _safe(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=df.index)

    def _s1(col):
        return _safe(col).shift(1)

    def _mom_s1(col, w, mp):
        return _safe(col).pct_change().rolling(w, min_periods=mp).sum().shift(1)

    def _diff_roll_s1(col, w, mp):
        return _safe(col).diff(1).rolling(w, min_periods=mp).sum().shift(1)

    # MOVE — shift(1): US rates vol, only final after US close
    move = _safe('MOVE Index  (R3)')
    df['move_level']  = move.shift(1)
    df['move_chg_1d'] = move.diff().shift(1)
    mv_mu = move.rolling(252, min_periods=60).mean()
    mv_sd = move.rolling(252, min_periods=60).std()
    df['move_zscore'] = ((move - mv_mu) / (mv_sd + 1e-9)).clip(-5, 5).shift(1)

    # VIX — shift(1): US 4pm close
    vix = _safe('^VIX')
    df['vix_level']  = vix.shift(1)
    vx_mu = vix.rolling(252, min_periods=60).mean()
    vx_sd = vix.rolling(252, min_periods=60).std()
    df['vix_zscore'] = ((vix - vx_mu) / (vx_sd + 1e-9)).clip(-5, 5).shift(1)

    # Additional vol indices — shift(1)
    for src, dst in [('VXN Index  (R2)', 'vxn_level'),
                     ('RVX Index  (L2)', 'rvx_level'),
                     ('OVX Index  (R3)', 'ovx_level'),
                     ('GVZ Index  (R2)', 'gvz_level'),
                     ('V2X Index  (L2)', 'v2x_level')]:
        df[dst] = _s1(src)

    # Equity — shift(1)
    for src, lbl in [('SPX Index  (R4)',  'spx'),
                     ('SX5E Index  (R4)', 'sx5e'),
                     ('MXWO Index  (R4)', 'mxwo'),
                     ('NDX Index  (L4)',  'ndx')]:
        df[f'{lbl}_mom_1m'] = _mom_s1(src, 21, 10)
        df[f'{lbl}_mom_3m'] = _mom_s1(src, 63, 30)

    # Commodities — shift(1)
    df['oil_mom_1m']    = _mom_s1('CL1 COMB Comdty  (R3)', 21, 10)
    df['oil_mom_3m']    = _mom_s1('CL1 COMB Comdty  (R3)', 63, 30)
    df['copper_mom_1m'] = _mom_s1('C01 Comdty  (R4)', 21, 10)
    df['opec_prod']     = _s1('OPECDALY Index  (R3)')
    df['natgas_mom_1m'] = _mom_s1('MUC1 Comdty  (L2)', 21, 10)

    # Breakeven inflation — shift(1): TIPS market (US)
    for bc in ['breakeven_5Y', 'breakeven_10Y']:
        be = _safe(bc)
        df[f'{bc}_level']  = be.shift(1)
        df[f'{bc}_chg_1m'] = be.diff(21).shift(1)
    df['breakeven_slope'] = (_safe('breakeven_10Y') - _safe('breakeven_5Y')).shift(1)

    # Credit spreads — shift(1): US credit market
    df['ig_spread'] = _s1('IG_spread')
    df['hy_spread'] = _s1('HY_spread')
    df['ig_chg_1m'] = _safe('IG_spread').diff(21).shift(1)
    df['hy_chg_1m'] = _safe('HY_spread').diff(21).shift(1)

    # Extra inflation — shift(1)
    df['cpi_nsa_level']  = _s1('CPURNSA Index  (L3)')
    df['pce_core_level'] = _s1('PCE CORE Index  (L2)')

    # Monthly macro — ffill+shift(1): same logic as v3
    for src, dst in [
        ('CPI YOY Index  (R1)',  'cpi_yoy'),
        ('PCE CRCH Index  (L1)', 'pce_core_chg'),
        ('NFP TCH Index  (L4)',  'nfp_change'),
        ('NAPMPMI Index  (R2)',  'pmi_us'),
        ('ECCPEMUY Index  (R1)', 'pmi_eur'),
        ('UMRTEMU Index  (R1)',  'unemp_eur'),
    ]:
        df[dst] = _safe(src).ffill().shift(1)

    # Country IBOR rates — contemporaneous (own-country daily fix)
    for country, (col3m, col6m) in COUNTRY_PRIMARY_IBOR.items():
        sr3m = _safe(col3m)
        df[f'{country}_ibor_level']      = sr3m
        df[f'{country}_ibor_chg_1d']     = sr3m.diff()
        df[f'{country}_ibor_mom_1m']     = sr3m.diff().rolling(21, min_periods=10).sum()
        sr6m = _safe(col6m) if col6m else pd.Series(np.nan, index=df.index)
        df[f'{country}_ibor_term_slope'] = sr6m - sr3m

    # Additional EURIBOR + NIBOR tenors — contemporaneous
    for src, dst in [
        ('EUR003M Index', 'eur_3m'),
        ('EUR001W Index', 'eur_1w'),
        ('NIBOR1M Index', 'nor_ibor_1m'),
        ('NIBOR6M Index', 'nor_ibor_6m'),
    ]:
        df[dst] = _safe(src)

    return df


print('add_macro_features_v4 defined')

### Curve helpers
Small helpers for the swap annuity, linear curve interpolation, and the carry/roll spread used downstream.

In [ ]:
def swap_annuity(S_pct, T_years):
    if pd.isna(S_pct) or S_pct <= 0.01:
        return np.nan
    S_semi = S_pct / 200
    n = 2 * T_years
    return (1 - (1 + S_semi) ** (-n)) / (S_pct / 100)


def interp_rate(country, T_target, df):
    pts = {}
    for m, T in MATURITY_YEARS.items():
        col = f'{country}_{m}'
        if col in df.columns:
            pts[T] = df[col]
    if not pts:
        return pd.Series(np.nan, index=df.index)
    ts = sorted(pts.keys())
    if T_target <= 0 or T_target <= ts[0]:
        return pd.Series(0.0, index=df.index)
    if T_target >= ts[-1]:
        return pts[ts[-1]]
    for i in range(len(ts) - 1):
        if ts[i] <= T_target <= ts[i + 1]:
            w = (T_target - ts[i]) / (ts[i + 1] - ts[i])
            return pts[ts[i]] * (1 - w) + pts[ts[i + 1]] * w
    return pd.Series(np.nan, index=df.index)


def carry_roll_spread(sec, df, country, maturity):
    T = MATURITY_YEARS.get(maturity, np.nan)
    if np.isnan(T):
        return pd.Series(np.nan, index=df.index)
    S_n   = df[sec]
    T_roll = max(0.0, T - 1.0)
    S_nm1  = (pd.Series(0.0, index=df.index) if T_roll == 0.0
              else interp_rate(country, T_roll, df))
    return S_n - S_nm1


print('Helper functions defined: swap_annuity, interp_rate, carry_roll_spread')

### Per-security feature set
Assemble the full feature matrix for one security: own-curve momentum/vol/level features, contemporaneous IBOR, the lagged global block, and the cross-market swap-rate features.

In [ ]:
def _features_for_security_v4(sec, df, country, maturity):
    """Full feature set for one security.

    Reads precomputed columns from df (add_macro_features_v4 applied, so exogenous
    series already carry shift(1)), and adds:
      - AR target lags chg_lag1..chg_lagTARGET_LAGS: backward-looking by
        construction (lag-k at t uses chg_1d[t-k]); picked up by leakage audit.
      - Extra vol indices: V2X, OVX, VXN, RVX, GVZ (shift(1) via precomputed)
      - Credit spreads: IG, HY (shift(1) via precomputed)
      - Energy: OPEC prod, natgas (shift(1) via precomputed)
      - Extra IBOR: EUR 1w/3m, NOR 1m/6m (contemporaneous via precomputed)
      - Cross-market swap rates: ALL other swap columns, shift(1) applied here.
        feature[t] = other_swap[t-1] — safe for all market timezones.
    """

    def _get(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=df.index)

    # ── Own-security features (contemporaneous) ───────────────────────────────
    level    = df[sec]
    chg_1d   = level.diff(1)

    # Explicit autoregressive target lags (lag-1..TARGET_LAGS of chg_1d).
    # Backward-looking by construction: lag-k at t uses chg_1d at t-k only.
    _tlags = {f'chg_lag{k}': chg_1d.shift(k) for k in range(1, TARGET_LAGS + 1)}

    mom_1m   = chg_1d.rolling(21,  min_periods=10).sum()
    mom_3m   = chg_1d.rolling(63,  min_periods=30).sum()
    mom_6m   = chg_1d.rolling(126, min_periods=60).sum()
    mom_12m  = chg_1d.rolling(252, min_periods=120).sum()
    mom_63d  = chg_1d.rolling(63,  min_periods=21).mean()
    dev_ma_3m  = level - level.rolling(63,  min_periods=21).mean()
    dev_ma_12m = level - level.rolling(252, min_periods=120).mean()
    rvol_20d   = chg_1d.rolling(20, min_periods=5).std()
    vol_mu     = rvol_20d.rolling(252, min_periods=60).mean()
    vol_sd     = rvol_20d.rolling(252, min_periods=60).std()
    vol_med    = rvol_20d.rolling(252, min_periods=60).median()
    vol_regime = (rvol_20d > vol_med).astype(float)
    vol_zscore = ((rvol_20d - vol_mu) / (vol_sd + 1e-9)).clip(-5, 5)
    lv_mu      = level.rolling(504, min_periods=120).mean()
    lv_sd      = level.rolling(504, min_periods=120).std()
    lv_zscore  = ((level - lv_mu) / (lv_sd + 1e-9)).clip(-5, 5)
    carry_roll = carry_roll_spread(sec, df, country, maturity)

    def _slope(lm, sm):
        lc, sc = f'{country}_{lm}', f'{country}_{sm}'
        if lc in df.columns and sc in df.columns:
            return df[lc] - df[sc]
        return pd.Series(np.nan, index=df.index)

    sofr_mom_1m = (df['SOFR_10Y'].diff(1).rolling(21, min_periods=10).sum()
                   if 'SOFR_10Y' in df.columns
                   else pd.Series(np.nan, index=df.index))

    base = {
        **_tlags,
        'carry_roll'         : carry_roll,
        'mom_63d'            : mom_63d,
        'mom_1m'             : mom_1m,
        'mom_3m'             : mom_3m,
        'mom_6m'             : mom_6m,
        'mom_12m'            : mom_12m,
        'dev_ma_3m'          : dev_ma_3m,
        'dev_ma_12m'         : dev_ma_12m,
        'vol_20d'            : rvol_20d,
        'vol_regime'         : vol_regime,
        'vol_zscore'         : vol_zscore,
        'level_zscore'       : lv_zscore,
        'slope_10_1'         : _slope('10Y', '1Y'),
        'slope_10_5'         : _slope('10Y', '5Y'),
        'slope_5_1'          : _slope('5Y',  '1Y'),
        'sofr_mom_1m'        : sofr_mom_1m,
        # country IBOR (contemporaneous)
        'ibor_level'         : _get(f'{country}_ibor_level'),
        'ibor_chg_1d'        : _get(f'{country}_ibor_chg_1d'),
        'ibor_mom_1m'        : _get(f'{country}_ibor_mom_1m'),
        'ibor_term_slope'    : _get(f'{country}_ibor_term_slope'),
        'swap_ibor_basis'    : level - _get(f'{country}_ibor_level'),
        # global vol (shift(1) baked into precomputed columns)
        'move_level'         : _get('move_level'),
        'move_zscore'        : _get('move_zscore'),
        'vix_level'          : _get('vix_level'),
        'vix_zscore'         : _get('vix_zscore'),
        'v2x_level'          : _get('v2x_level'),
        'vxn_level'          : _get('vxn_level'),
        'ovx_level'          : _get('ovx_level'),
        'gvz_level'          : _get('gvz_level'),
        # equity (shift(1))
        'spx_mom_1m'         : _get('spx_mom_1m'),
        'spx_mom_3m'         : _get('spx_mom_3m'),
        'sx5e_mom_1m'        : _get('sx5e_mom_1m'),
        'sx5e_mom_3m'        : _get('sx5e_mom_3m'),
        'mxwo_mom_1m'        : _get('mxwo_mom_1m'),
        'ndx_mom_1m'         : _get('ndx_mom_1m'),
        # commodities (shift(1))
        'oil_mom_1m'         : _get('oil_mom_1m'),
        'oil_mom_3m'         : _get('oil_mom_3m'),
        'copper_mom_1m'      : _get('copper_mom_1m'),
        'opec_prod'          : _get('opec_prod'),
        'natgas_mom_1m'      : _get('natgas_mom_1m'),
        # breakeven + credit (shift(1))
        'be5y_level'         : _get('breakeven_5Y_level'),
        'be10y_level'        : _get('breakeven_10Y_level'),
        'be_slope'           : _get('breakeven_slope'),
        'be5y_chg_1m'        : _get('breakeven_5Y_chg_1m'),
        'ig_spread'          : _get('ig_spread'),
        'hy_spread'          : _get('hy_spread'),
        'ig_chg_1m'          : _get('ig_chg_1m'),
        'hy_chg_1m'          : _get('hy_chg_1m'),
        # monthly macro (ffill+shift(1))
        'cpi_yoy'            : _get('cpi_yoy'),
        'pce_core_chg'       : _get('pce_core_chg'),
        'nfp_change'         : _get('nfp_change'),
        'pmi_us'             : _get('pmi_us'),
        'pmi_eur'            : _get('pmi_eur'),
        'unemp_eur'          : _get('unemp_eur'),
        'cpi_nsa_level'      : _get('cpi_nsa_level'),
        'pce_core_level'     : _get('pce_core_level'),
        # extra IBOR tenors (contemporaneous)
        'eur_3m'             : _get('eur_3m'),
        'eur_1w'             : _get('eur_1w'),
        'nor_ibor_1m'        : _get('nor_ibor_1m'),
        'nor_ibor_6m'        : _get('nor_ibor_6m'),
    }

    # Cross-market swap rates — ALL other swap cols, shift(1)
    # feature[t] = other_swap_close[t-1], safe for all timezones
    xm = {}
    other_swaps = sorted(c for c in df.columns if _SWAP_PAT.match(c) and c != sec)
    for col in other_swaps:
        fn = 'xm_' + re.sub(r'[^a-zA-Z0-9]', '_', col).strip('_')
        xm[fn + '_lv'] = df[col].shift(1)
        xm[fn + '_ch'] = df[col].diff(1).shift(1)

    # ── Norway-specific features — gated to country=='NOR' (Part B) ──────────
    # Read precomputed nor_* columns (timing baked in by add_norway_features_v4)
    # plus the swap-govvie spread for the security's own tenor (contemporaneous).
    nor = {}
    if country == 'NOR':
        for fn in sorted(c for c in df.columns if c.startswith('nor_')):
            nor[fn] = df[fn]
        gcol = f'nor_govt_{maturity.lower()}'
        nor['swap_govt_spread'] = ((level - df[gcol]) if gcol in df.columns
                                   else pd.Series(np.nan, index=df.index))

    return pd.DataFrame({**base, **xm, **nor}, index=df.index)


print('_features_for_security_v4 defined')
print(f'  target lags: {TARGET_LAGS} (chg_lag1..chg_lag{TARGET_LAGS})')
print(f'  base features: ~{63 + TARGET_LAGS}  +  cross-market: ~2 × (|UNIVERSE|-1)')

### Build the modelling panel
Stack the per-security features into one long panel and attach the forecasting target — the h-day-ahead change in the swap level.

In [ ]:
META_COLS = {'date', 'security', 'y', 'level'}


def build_panel_v4(df_raw, securities, h):
    """Stacked per-security panel. Target: outright h-day change (not residual)."""
    rows = []
    for sec in securities:
        if sec not in df_raw.columns:
            continue
        country, maturity = sec.rsplit('_', 1)
        level = df_raw[sec]
        feats = _features_for_security_v4(sec, df_raw, country, maturity)
        y     = level.shift(-h) - level
        sec_df             = feats.reset_index().rename(columns={'index': 'date',
                                                                  df_raw.index.name or 'Date': 'date'})
        # handle named index
        if 'date' not in sec_df.columns:
            sec_df.insert(0, 'date', feats.index)
        sec_df['security'] = sec
        sec_df['y']        = y.values
        sec_df['level']    = level.values
        rows.append(sec_df)

    if not rows:
        return pd.DataFrame()
    panel = pd.concat(rows, ignore_index=True)
    panel = panel.dropna(subset=['y'])
    panel = panel.sort_values(['date', 'security']).reset_index(drop=True)
    return panel


print('build_panel_v4 defined')

### Apply the macro/global features
Run the macro feature builder on the raw panel and report which columns were added. The builder derives columns from raw source series, so re-running recomputes identical values rather than double-applying.

In [ ]:
_cols_before = set(df_raw.columns)
t0 = time.time()
df_raw = add_macro_features_v4(df_raw)
print(f'df_raw augmented in {time.time()-t0:.1f}s  →  shape: {df_raw.shape}')

new_cols = sorted(c for c in df_raw.columns if c not in _cols_before)
print(f'\nDerived macro/global columns added ({len(new_cols)}):')
print(f'  {"Column":<35s}  {"n_obs":>6s}')
print('  ' + '-' * 44)
for c in new_cols:
    n = df_raw[c].notna().sum()
    print(f'  {c:<35s}  {n:6d}')


# Norway nor_* columns were added in Part B (above) and are preserved here.
_n_nor = sum(1 for c in df_raw.columns if c.startswith('nor_'))
print(f'\nNorway nor_* feature columns present after augmentation: {_n_nor}')

### Inspect one security's features
Print the feature matrix for the pilot security with a timing annotation per column, as an eyeball check that no exogenous feature is contemporaneous or forward-looking.

In [ ]:
# ── Step 0a: Print sample feature matrix with lags ───────────────────────────
# For one security, show all feature columns with their timing/lag annotation.
# Eyeball check: nothing contemporaneous-or-future for exogenous series.

_c, _m    = SMOKE_SEC.rsplit('_', 1)
_feat_df  = _features_for_security_v4(SMOKE_SEC, df_raw, _c, _m)
# Sample the last date where THIS security actually trades (own-security features
# populated). The last calendar row lands past the swap's data end, where only
# ffill-macro / cross-market features survive and every own column is NaN.
_last_idx = df_raw[SMOKE_SEC].dropna().index[-1]
_row      = _feat_df.loc[_last_idx]

print(f'=== Feature matrix for {SMOKE_SEC} ===')
print(f'  Shape: {_feat_df.shape[0]} dates × {_feat_df.shape[1]} features')
print(f'  Sample date: {_last_idx}')
print()

# Timing annotations
def _timing(fname):
    if fname.startswith('chg_lag'):
        return 'backward AR lag (chg_1d shifted k days; own security)'
    if fname in ('carry_roll','mom_63d','mom_1m','mom_3m','mom_6m','mom_12m',
                 'dev_ma_3m','dev_ma_12m','vol_20d','vol_regime','vol_zscore',
                 'level_zscore','slope_10_1','slope_10_5','slope_5_1'):
        return 'contemporaneous (own security daily close)'
    if fname in ('ibor_level','ibor_chg_1d','ibor_mom_1m','ibor_term_slope',
                 'swap_ibor_basis','sofr_mom_1m','eur_3m','eur_1w',
                 'nor_ibor_1m','nor_ibor_6m'):
        return 'contemporaneous (IBOR / same-session fix)'
    if any(fname.startswith(p) for p in ('move_','vix_','v2x_','vxn_','rvx_',
                                          'ovx_','gvz_')):
        return 'shift(1): vol index, prior close'
    if any(fname.startswith(p) for p in ('spx_','sx5e_','mxwo_','ndx_')):
        return 'shift(1): equity close, prior day'
    if any(fname.startswith(p) for p in ('oil_','copper_','opec_','natgas_')):
        return 'shift(1): commodity close, prior day'
    if any(fname.startswith(p) for p in ('be5y_','be10y_','be_slope','ig_','hy_')):
        return 'shift(1): TIPS/credit market, prior day'
    if any(fname.startswith(p) for p in ('cpi_','pce_','nfp_','pmi_','unemp_')):
        return 'ffill+shift(1): monthly release, prior value'
    if fname.startswith('xm_'):
        return 'shift(1): cross-market swap, prior close'
    if fname.startswith('nor_'):
        if any(fname.startswith(p) for p in ('nor_eurnok', 'nor_usdnok', 'nor_i44')):
            return 'shift(1): NOK FX / krone index, prior close'
        if any(fname.startswith(p) for p in ('nor_kpi', 'nor_kpijae', 'nor_unemp')):
            return 'ffill+shift(1): SSB monthly release, prior value'
        return 'contemporaneous (NOK official / administered same-session)'
    if fname == 'swap_govt_spread':
        return 'contemporaneous (swap − govvie, same-session)'
    return '?'

n_xm   = sum(1 for f in _row.index if f.startswith('xm_'))
n_base = len(_row) - n_xm
print(f'  {"Feature":<42s}  {"Value":>10s}  Timing')
print('  ' + '-' * 90)
for feat, val in _row.items():
    if feat.startswith('xm_') and n_xm > 6:
        continue  # print summary line instead of all cross-market
    val_str = f'{val:.4f}' if pd.notna(val) else 'NaN'
    print(f'  {feat:<42s}  {val_str:>10s}  {_timing(feat)}')
if n_xm > 6:
    non_nan_xm = sum(1 for f,v in _row.items() if f.startswith('xm_') and pd.notna(v))
    print(f'  {"[xm_* cross-market features]":<42s}  '
          f'{non_nan_xm:>10d} non-NaN  shift(1): cross-market swap, prior close')
print(f'\n  Base features: {n_base}   Cross-market features: {n_xm}   Total: {len(_row)}')
print()
print('[LEAKAGE CHECK] All exogenous features (VIX, equity, commodity, credit, cross-market)')
print('  are lagged 1 period. Own-security features are contemporaneous (daily close).')
print('  Monthly macro series were already ffill+shift(1) in v3; unchanged.')

### Leakage audit
Independently re-derive each feature from data available only up to time *t* and compare it against the stored panel value, so any look-ahead leak fails loudly before the heavy runs.

**Scope note.** The audit fully re-derives the own-curve and cross-market features point-in-time. The precomputed macro/global columns are validated as already-lagged stored values rather than being re-derived independently — a known scope boundary, not a bug.

In [ ]:
# ── Step 0b: Leakage audit on full expanded feature set ──────────────────────
# Recompute features from df_raw.loc[<=t] and compare with stored panel values.
# Audits ALL columns including cross-market swap feats.

def leakage_audit_v4(panel, df_raw, n_samples=30, rng_seed=config.JULIA_SEED, skip_first=500):
    _META     = {'date', 'security', 'y', 'level'}
    feat_cols = [c for c in panel.columns if c not in _META]

    eligible = panel.iloc[skip_first:].copy()
    sample   = eligible.sample(n=min(n_samples, len(eligible)), random_state=rng_seed)

    TOL          = 1e-6
    fail_counts  = {c: 0 for c in feat_cols}
    check_counts = {c: 0 for c in feat_cols}

    for _, row in sample.iterrows():
        t       = row['date']
        sec     = row['security']
        country, maturity = sec.rsplit('_', 1)
        past = df_raw.loc[df_raw.index <= t]
        if sec not in past.columns or past[sec].isna().all():
            continue
        feats_at_t = _features_for_security_v4(sec, past, country, maturity).iloc[-1]

        for col in feat_cols:
            stored = row[col]
            recomp = feats_at_t.get(col, np.nan)
            check_counts[col] += 1
            if pd.isna(stored) and pd.isna(recomp):
                continue
            if pd.isna(stored) != pd.isna(recomp):
                fail_counts[col] += 1
                continue
            if abs(float(stored) - float(recomp)) > TOL:
                fail_counts[col] += 1

    n_fail = sum(1 for c in feat_cols if fail_counts[c] > 0)
    n_skip = sum(1 for c in feat_cols if check_counts[c] == 0)
    n_pass = len(feat_cols) - n_fail - n_skip

    print(f'=== Leakage audit v4 — {len(feat_cols)} features, '
          f'{n_samples} sample rows ===\n')

    for col in feat_cols:
        if fail_counts[col] > 0:
            n = check_counts[col]
            print(f'  FAIL  {col}  {n-fail_counts[col]}/{n} OK  '
                  f'← {fail_counts[col]} mismatch(es)')

    print(f'\n  Summary: {n_pass} PASS / {n_fail} FAIL / {n_skip} SKIP')
    if n_fail == 0:
        print(f'\n[PASS] All {n_pass} checked features leakage-free.')
        print(f'  VIX/MOVE/equity/commodity: shift(1) in add_macro_features_v4.')
        print(f'  Cross-market swap rates: shift(1) in _features_for_security_v4.')
    else:
        print(f'\n[FAIL] {n_fail} feature(s) have mismatches — fix before full run.')

    return n_fail == 0


_pilot_panel = build_panel_v4(df_raw, [SMOKE_SEC], H)
print(f'Pilot panel for audit: {_pilot_panel.shape}  '
      f'({_pilot_panel.shape[1]-len(META_COLS)} feature cols)')
print()
_audit_ok = leakage_audit_v4(_pilot_panel, df_raw, n_samples=30)
assert _audit_ok, 'Leakage audit failed — fix before proceeding'

## Metrics contract — shared long-CSV schema

Every results row — walk-forward **and** block-CV/LORO, **train** and **oos** — is produced by the
single function `compute_metrics_row(...)` and written to **`v5_metrics_long.csv`**. The pooled
notebook must emit the **identical** shared columns (it may add pooled-only columns; it must never
rename or drop a shared one). One row per `(security, tenor, horizon, fold/block, scheme, sample)`.

**Identifier / provenance columns**

| column | dtype | meaning |
|---|---|---|
| `notebook` | str | `'v5'` |
| `run_ts` | str | ISO timestamp of the run |
| `model_kind` | str | `'per_security'` here; `'pooled'` in the pooled notebook |
| `is_pooled` | bool | `False` here, `True` in the pooled notebook |
| `validation_scheme` | str | `'walk_forward'` \| `'block_cv'` \| `'loro'` |
| `target_kind` | str | `'level_change'` (default) \| `'excess_hpr'` (optional long-horizon target) |
| `security` | str | e.g. `NOR_10Y` |
| `country` | str | e.g. `NOR` |
| `tenor` | str | e.g. `10Y` |
| `horizon` | int | h (business days) |
| `fold` | str | fold/block name (e.g. `Hiking`, `Block_ZIRP`) |
| `regime` | str | `GFC`\|`ZIRP`\|`COVID`\|`Hiking` |
| `sample` | str | `'train'` \| `'oos'` |
| `config_hash` | str | hash of the frozen config + feature spec |
| `feature_count` | int | #features actually fed to HTBoost (post-PCA) |

**Metric columns** (random-walk benchmark forecasts the change as `ŷ_RW = 0`, so `mse_rw = mean(y²)`)

| column | dtype | meaning |
|---|---|---|
| `n_obs` | int | scored observations |
| `dir_acc` | float | directional accuracy, `mean(sign(y)==sign(ŷ))` |
| `r2_raw` | float | `sklearn.r2_score(y, ŷ)` |
| `mse_model` | float | `mean((y−ŷ)²)` |
| `mse_rw` | float | `mean(y²)` |
| `ct_r2_oos` | float | **Campbell–Thompson** OOS-R² `= 1 − mse_model/mse_rw` |
| `cw_stat`,`cw_pval` | float | **Clark–West** vs RW (HAC/Newey–West, lag≈H−1), one-sided (model better) |
| `dmh_stat`,`dmh_pval` | float | **Diebold–Mariano–Harvey** vs RW (HLN small-sample corr., HAC lag≈H−1), one-sided |
| `pt_stat`,`pt_pval` | float | **Pesaran–Timmermann** directional, one-sided |
| `binom_pval` | float | one-sided binomial vs 0.5 |
| `n_eff` | int | effective sample size for overlapping returns `≈ n/H` (binomial/PT caveat) |

**MTC columns** (added per family, **never pooled across families**)

| column | dtype | meaning |
|---|---|---|
| `reject_bonferroni` | bool | survives Bonferroni within its MTC family |
| `reject_fdr_bh` | bool | survives BH-FDR within its MTC family |
| `mtc_N` | int | family size used |
| `mtc_family` | str | `walk_forward:{horizon×tenor×regime}` or `{scheme}:{horizon×tenor×block}` |

Walk-forward and block-CV/LORO are corrected in **separate MTC families** (they test
forecastability vs learnability — different hypotheses).

In [ ]:
# ── Metrics module — single source of truth for every results row ──
# RW benchmark forecasts the h-day change as 0 ⇒ rw error = y, mse_rw = mean(y²).
# Overlapping h>1 returns are handled with HAC/Newey–West variance (lag = H−1) in
# the Clark–West and Diebold–Mariano–Harvey tests, and an effective-sample-size
# note (n_eff ≈ n/H) for the binomial / Pesaran–Timmermann directional tests.

def _hac_mean_tstat(z, lag):
    """t-stat of mean(z)=0 with Newey–West HAC variance (lag≥0). Returns (mean, tstat, n)."""
    z = np.asarray(z, dtype=float)
    z = z[np.isfinite(z)]
    n = z.size
    if n < 5 or np.allclose(z, z[0]):
        return (float(np.mean(z)) if n else np.nan, np.nan, n)
    X = np.ones((n, 1))
    try:
        res = sm.OLS(z, X).fit(cov_type='HAC', cov_kwds={'maxlags': max(int(lag), 0)})
        return float(res.params[0]), float(res.tvalues[0]), n
    except Exception:
        return float(np.mean(z)), np.nan, n


def clark_west(y, yhat, H):
    """Clark–West vs RW(=0). f_t = y² − (y−ŷ)² + ŷ²; one-sided p (model better)."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[m], yhat[m]
    if y.size < 5:
        return (np.nan, np.nan)
    f = y**2 - (y - yhat)**2 + yhat**2
    _, t, _ = _hac_mean_tstat(f, H - 1)
    if not np.isfinite(t):
        return (np.nan, np.nan)
    return (t, float(1 - norm.cdf(t)))           # one-sided: H1 = model beats RW


def dm_harvey(y, yhat, H):
    """Diebold–Mariano with Harvey–Leybourne–Newbold small-sample correction, HAC lag=H−1.
    Loss differential d_t = e_rw² − e_m² = y² − (y−ŷ)²  (d>0 ⇒ model better). One-sided p."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[m], yhat[m]
    n = y.size
    if n < 5:
        return (np.nan, np.nan)
    d = y**2 - (y - yhat)**2
    _, dm, _ = _hac_mean_tstat(d, H - 1)
    if not np.isfinite(dm):
        return (np.nan, np.nan)
    h = max(int(H), 1)
    hln = np.sqrt(max((n + 1 - 2 * h + h * (h - 1) / n) / n, 1e-12))   # HLN factor
    dm_star = dm * hln
    from scipy.stats import t as _t
    return (float(dm_star), float(_t.sf(dm_star, df=n - 1)))           # one-sided: model better


def pesaran_timmermann(y, yhat):
    """Pesaran–Timmermann (1992) directional-accuracy test. One-sided p (predictability)."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[m], yhat[m]
    n = y.size
    if n < 5:
        return (np.nan, np.nan)
    Dy = (y > 0).astype(float)
    Dx = (yhat > 0).astype(float)
    P = float(np.mean(Dy == Dx))
    Py, Px = float(np.mean(Dy)), float(np.mean(Dx))
    Pstar = Py * Px + (1 - Py) * (1 - Px)
    var_P = Pstar * (1 - Pstar) / n
    var_Pstar = (((2 * Py - 1) ** 2) * Px * (1 - Px) / n
                 + ((2 * Px - 1) ** 2) * Py * (1 - Py) / n
                 + 4 * Py * Px * (1 - Py) * (1 - Px) / (n ** 2))
    denom = var_P - var_Pstar
    if denom <= 0:
        return (np.nan, np.nan)
    pt = (P - Pstar) / np.sqrt(denom)
    return (float(pt), float(1 - norm.cdf(pt)))


def _score(y, yhat):
    """DirAcc/R²/n helper used by the smoke and Part-C prints."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    if m.sum() < 5:
        return None
    return {'dir_acc': float(np.mean(np.sign(y[m]) == np.sign(yhat[m]))),
            'r2': float(r2_score(y[m], yhat[m])), 'n_obs': int(m.sum())}


def config_hash(cfg, extra=''):
    payload = json.dumps({k: cfg.get(k) for k in sorted(cfg)} if isinstance(cfg, dict) else cfg,
                         sort_keys=True, default=str) + '|' + str(extra)
    return hashlib.md5(payload.encode()).hexdigest()[:12]


# Frozen list of shared schema columns (the pooled-comparability contract).
SHARED_COLS = [
    'notebook', 'run_ts', 'model_kind', 'is_pooled', 'validation_scheme', 'target_kind',
    'security', 'country', 'tenor', 'horizon', 'fold', 'regime', 'sample',
    'config_hash', 'feature_count',
    'n_obs', 'dir_acc', 'dir_coverage', 'r2_raw', 'mse_model', 'mse_rw', 'mae', 'ct_r2_oos',
    'cw_stat', 'cw_pval', 'dmh_stat', 'dmh_pval', 'pt_stat', 'pt_pval',
    'binom_pval', 'n_eff',
]


def compute_metrics_row(y, yhat, H, meta):
    """Build ONE shared-schema row. `meta` carries the identifier columns.
    Used identically by the walk-forward, block-CV and LORO paths (and replicable
    in the pooled notebook)."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    yy, yh = y[m], yhat[m]
    n = int(yy.size)
    row = {c: meta.get(c) for c in SHARED_COLS if c in meta}
    row.update({'notebook': NOTEBOOK_TAG, 'run_ts': RUN_TS,
                'model_kind': 'per_security', 'is_pooled': False,
                'horizon': int(H)})
    row.update({k: meta[k] for k in
                ('validation_scheme', 'target_kind', 'security', 'country', 'tenor',
                 'fold', 'regime', 'sample', 'config_hash', 'feature_count') if k in meta})
    if n < 5:
        for c in ('n_obs', 'dir_acc', 'dir_coverage', 'r2_raw', 'mse_model', 'mse_rw', 'mae', 'ct_r2_oos',
                  'cw_stat', 'cw_pval', 'dmh_stat', 'dmh_pval', 'pt_stat', 'pt_pval',
                  'binom_pval', 'n_eff'):
            row[c] = np.nan
        row['n_obs'] = n
        return row
    mse_model = float(np.mean((yy - yh) ** 2))
    mse_rw    = float(np.mean(yy ** 2))
    mae       = float(np.mean(np.abs(yy - yh)))
    dir_acc   = float(np.mean(np.sign(yy) == np.sign(yh)))
    # Directional coverage: share of SCORED obs with a nonzero forecast (≈1 for the
    # feature-based models; <1 for the rule-based benchmarks whenever they fall back to
    # the no-change forecast). Reported alongside dir_acc per the evaluation design.
    dir_coverage = float(np.mean(np.sign(yh) != 0))
    cw_s, cw_p   = clark_west(yy, yh, H)
    dm_s, dm_p   = dm_harvey(yy, yh, H)
    pt_s, pt_p   = pesaran_timmermann(yy, yh)
    k = int(np.sum(np.sign(yy) == np.sign(yh)))
    binom_p = binomtest(k, n, p=0.5, alternative='greater').pvalue
    row.update({
        'n_obs': n, 'dir_acc': dir_acc, 'dir_coverage': dir_coverage,
        'r2_raw': float(r2_score(yy, yh)),
        'mse_model': mse_model, 'mse_rw': mse_rw, 'mae': mae,
        'ct_r2_oos': float(1 - mse_model / mse_rw) if mse_rw > 0 else np.nan,
        'cw_stat': cw_s, 'cw_pval': cw_p, 'dmh_stat': dm_s, 'dmh_pval': dm_p,
        'pt_stat': pt_s, 'pt_pval': pt_p, 'binom_pval': float(binom_p),
        'n_eff': int(max(1, round(n / max(int(H), 1)))),
    })
    return row


print('Metrics module defined: compute_metrics_row, clark_west, dm_harvey,')
print('  pesaran_timmermann, _score, config_hash;  SHARED_COLS =', len(SHARED_COLS), 'columns')

### Cross-market PCA (fit on training rows only)
Compress the large cross-market `xm_*` block to principal components, fitting the standardizer and PCA on training rows only and applying them to the test rows. The number of components is the **smaller of** {count reaching the 95% (`XM_PCA_VAR`) train-variance target, `XM_PCA_KMAX`=12}. In practice the **12-component cap binds first**: it is reached before the 95% target, so the components retain only **roughly 70%** of cross-market variance (the smoke fold shows cum≈0.70, not 0.95). The cap is an *a-priori, uniform* choice applied identically to every tenor/horizon/fold — it is **not tuned**, and the realised cumulative explained-variance ratio is logged per fold as `xm_pca_evr`. Living in the fit path (not the feature builder) keeps the leakage audit valid.

In [ ]:
# ── PCA compression of the cross-market xm_* block — FIT ON TRAIN ONLY ──
# Splits the feature matrix into xm_* (cross-market) vs the rest. Standardises and
# PCA-fits ONLY on the training rows, then transforms train AND test with the SAME
# fitted objects, so test data never touches the PCA fit. The xm_* columns are
# replaced by k components (xmpca_01..xmpca_kk). Pre-committed rule:
#   k = smallest #components explaining ≥ XM_PCA_VAR of TRAIN variance, capped at XM_PCA_KMAX.
# This is in the FIT PATH — deliberately NOT in _features_for_security_v4 (the leakage
# audit recomputes that function from df_raw.loc[<=t] and cannot validate a fold-fit
# transform; putting PCA there would either break the audit or leak).

# Shared cross-market PCA (fit on TRAIN only) — single source of truth.
from src.features.pca import _xm_split, fit_transform_xm_pca


print('PCA fit-path helper defined: fit_transform_xm_pca (fit on TRAIN rows only)')

### HTBoost bridge
Helpers that sanitize the feature frame, build an HTBoost parameter object from the frozen config (threading only fields the installed build actually exposes), and fit/predict through Julia.

In [ ]:
# ── juliacall bridge helpers ────────────────────────────────────────────
# Bridge invariants: 4-arg HTBdata(y, x, param, dates); NO fnames=;
# randomizecv=False; overlap=H-1; 'lambda' threaded via a dict (Python keyword).
# The frozen config (lambda/depth/nfold/ntrees/modality) is threaded through the
# SAME dict, filtered against the REAL HTBparam field names introspected at runtime
# (names are not assumed). Optional feature-importance extraction via HTBrelevance
# (names come from the DataFrame columns automatically).

def _sanitize_cols(df):
    rmap = {c: re.sub(r'[^a-zA-Z0-9]', '_', str(c)).strip('_') for c in df.columns}
    return df.rename(columns=rmap), rmap


def _dedup_cols(df):
    seen, new_cols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c] += 1; new_cols.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 0; new_cols.append(c)
    df.columns = new_cols
    return df


def prepare_x_v5(df):
    df, _ = _sanitize_cols(df.copy())
    df    = _dedup_cols(df)
    df    = (df.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float64))
    return df


def _to_julia_dates(date_series):
    ds = [d.strftime('%Y-%m-%d') for d in pd.to_datetime(date_series)]
    return jl.seval('x -> Date.(x, dateformat"y-m-d")')(ds)


# Introspect the REAL HTBparam field names once (names are not assumed).
VALID_HTB_FIELDS = set(str(f) for f in jl.seval('collect(string.(fieldnames(typeof(HTBparam()))))'))
for _need in ('lambda', 'depth', 'nfold', 'ntrees', 'modality', 'loss', 'overlap', 'randomizecv'):
    assert _need in VALID_HTB_FIELDS, f'HTBparam has no field {_need!r} — bridge needs it'
print(f'HTBparam exposes {len(VALID_HTB_FIELDS)} fields; threading only valid keys.')


def _build_param(cfg, H):
    """Construct HTBparam from a frozen-config dict, threading only valid fields."""
    kw = dict(loss=cfg.get('loss', LOSS), modality=cfg.get('modality', MODALITY),
              nfold=int(cfg.get('nfold', NFOLD_INTERNAL)), randomizecv=False,
              overlap=int(H - 1), verbose='Off')
    if cfg.get('lam') is not None:
        kw['lambda'] = float(cfg['lam'])
    if cfg.get('depth') is not None:
        kw['depth'] = int(cfg['depth'])
    if cfg.get('ntrees') is not None:
        kw['ntrees'] = int(cfg['ntrees'])
    bad = [k for k in kw if k not in VALID_HTB_FIELDS and k != 'verbose']
    assert not bad, f'unknown HTBparam keys: {bad}'
    return jl.HTBparam(**kw)


# ── τ^w smoothness diagnostic helpers (HTBoost) ──────────────────────────
# τ^w is the importance-weighted geometric-mean smoothing parameter (paper eq. 19
# + footnote 8): per-feature avgtau capped at 40, then a geometric mean weighted by
# HTBrelevance importance. It is a MODEL DIAGNOSTIC (curve smoothness), never an
# accuracy metric — it is kept OUT of the shared htb_metrics schema by design.
TAU_W_CAP   = 40.0
# Interpretive bands (HTBoost tutorial thresholds), upper-inclusive cutoffs.
TAU_W_BANDS = ((7.0, 'strong'), (15.0, 'mild'), (25.0, 'weak'))   # >25 → 'none'


def tau_w_band(tau):
    """Map a scalar τ^w to its interpretive smoothness band:
    ≤7 'strong', 7–15 'mild', 15–25 'weak', >25 'none' (≈ hard splits / no smoothness).
    Returns 'na' for a missing/non-finite τ^w."""
    if tau is None or not np.isfinite(tau):
        return 'na'
    for hi, lab in TAU_W_BANDS:
        if tau <= hi:
            return lab
    return 'none'


def _weighted_tau_scalar(importance, avgtau, cap=TAU_W_CAP):
    """Aggregate scalar τ^w per eq. (19) + footnote 8: cap each per-feature avgtau at
    `cap`, then take the importance-weighted GEOMETRIC mean. NaN if undefined."""
    w = np.asarray(importance, dtype=float)
    t = np.minimum(np.asarray(avgtau, dtype=float), cap)
    m = np.isfinite(w) & np.isfinite(t) & (w > 0) & (t > 0)
    if not m.any():
        return float('nan')
    w, t = w[m], t[m]
    return float(np.exp(np.sum(w * np.log(t)) / np.sum(w)))


# τ^w field resolution — the installed HybridTreeBoosting build may name the
# HTBweightedtau columns differently from the tutorial, so we resolve them from the
# REAL returned keys (case-insensitive substring) instead of hard-coding names.
# Pinned below to the installed build's schema (confirmed via the run-once PROBE
# cell); overrides win over auto-resolution. Re-run the PROBE and edit the dict if
# the build ever changes.
HTB_TAU_FIELD_OVERRIDES = {
    # Pinned to the installed HybridTreeBoosting build's HTBweightedtau schema
    # (confirmed via the PROBE cell). gavgtau = geometric importance-weighted mean
    # = paper eq.19 + footnote 8 = the reported τ^w (NOT the arithmetic avgtau).
    'tau_scalar':     'gavgtau',   # scalar τ^w (read directly)
    'tau_table':      'df',        # per-feature DataFrame field
    'feature_col':    'feature',   # column inside df
    'importance_col': 'importance',# column inside df
    'smoothness_col': 'avgtau',    # per-feature smoothness column inside df
}
_TAU_FIELD_CANDIDATES = {
    'feature':    ['feature', 'fname', 'fnames', 'name', 'names', 'var'],
    'importance': ['importance', 'relevance', 'fi', 'weight', 'rel'],
    'smoothness': ['avgtau', 'gavgtau', 'meantau', 'avg_tau', 'tau', 'smooth'],
}
_TAU_SCALAR_CANDIDATES = ['gavgtau', 'tau_w', 'wtau', 'weightedtau', 'taubar', 'avgtau_w']


def _resolve_field(keys, candidates, override=None):
    """Pick the key matching an ordered candidate list. An explicit override wins
    (returned only if actually present). Otherwise: exact case-insensitive match in
    candidate order, then substring match in candidate order, preferring a non-
    'sorted'/'index' variant so per-feature columns stay row-aligned. None if none match."""
    keys = list(keys)
    if override is not None:
        return override if override in keys else None
    low = {k.lower(): k for k in keys}
    for cand in candidates:                                   # exact (case-insensitive)
        if cand.lower() in low:
            return low[cand.lower()]
    matches = []                                              # substring, candidate order
    for cand in candidates:
        for k in keys:
            if cand.lower() in k.lower() and k not in matches:
                matches.append(k)
    if not matches:
        return None
    plain = [k for k in matches
             if not any(s in k.lower() for s in ('sort', 'indx', 'index', 'idx'))]
    return (plain or matches)[0]


def _extract_weightedtau(wt):
    """Resolve (tau_w_scalar, per_feature_DataFrame) from an HTBweightedtau return
    object using the PINNED HTB_TAU_FIELD_OVERRIDES (confirmed against the installed
    build via the PROBE cell). The scalar τ^w is read DIRECTLY from the geometric
    importance-weighted-mean field (`gavgtau` = paper eq.19 + fn.8) — never the
    arithmetic `avgtau`, never recomputed. The per-feature table is pulled from the
    `df` field (a Julia DataFrame). `_weighted_tau_scalar` is used ONLY as a cross-
    check: if the paper-formula reconstruction disagrees with the package scalar, a
    one-line warning is printed (we still report the package value). Raises if the
    scalar field is absent (the caller logs the keys and returns None)."""
    try:
        wt_keys = [str(f) for f in jl.seval('x -> collect(string.(keys(x)))')(wt)]
    except Exception:
        wt_keys = [str(f) for f in jl.seval('x -> collect(string.(propertynames(x)))')(wt)]
    ov = HTB_TAU_FIELD_OVERRIDES
    k_scalar = ov.get('tau_scalar') or _resolve_field(wt_keys, _TAU_SCALAR_CANDIDATES)
    if not k_scalar or k_scalar not in wt_keys:
        raise KeyError(f'τ^w scalar field not found (keys={wt_keys})')
    tau_w = float(getattr(wt, k_scalar))
    tau_table = None
    k_table = ov.get('tau_table')
    f_col = ov.get('feature_col', 'feature')
    i_col = ov.get('importance_col', 'importance')
    s_col = ov.get('smoothness_col', 'avgtau')
    if k_table and k_table in wt_keys:
        _dfj  = getattr(wt, k_table)
        _pull = jl.seval('(d, c) -> collect(d[!, Symbol(c)])')
        feats = [str(x) for x in _pull(_dfj, f_col)]
        imp_w = np.asarray(_pull(_dfj, i_col), dtype=float)
        avgt  = np.asarray(_pull(_dfj, s_col), dtype=float)
        tau_table = pd.DataFrame({'feature': feats, 'importance': imp_w, 'avgtau': avgt})
        _recon = _weighted_tau_scalar(imp_w, avgt)     # cap-40 weighted geo-mean cross-check
        if (np.isfinite(_recon) and np.isfinite(tau_w)
                and abs(_recon - tau_w) > max(0.05, 0.05 * tau_w)):
            print(f'    [warn] τ^w cross-check: package {k_scalar}={tau_w:.3f} vs '
                  f'eq.19 reconstruction={_recon:.3f} disagree (reporting {k_scalar})')
    return tau_w, tau_table


def fit_htboost_v5(y_train, x_train_df, dates_series, x_test_df, cfg, H,
                   want_importance=False, want_tau=False):
    """Fit one HTBoost model with a frozen config. Returns the 6-tuple
    (output, yhat_train, yhat_test_or_None, importance_dict_or_None,
     tau_w_or_None, tau_table_or_None).

    importance : {feature: relevance} from HTBrelevance, when want_importance.
    tau_w      : scalar importance-weighted smoothness τ^w (eq. 19; per-feature
                 avgtau capped at 40, geometric mean), when want_tau.
    tau_table  : per-feature DataFrame[feature, importance, avgtau] from
                 HTBweightedtau, when want_tau.
    Both τ outputs are a model DIAGNOSTIC, not an accuracy metric, and are returned
    as None on failure (a warning is logged; a single fold is never aborted)."""
    assert int(H - 1) >= 0
    x_tr     = prepare_x_v5(x_train_df)
    x_jl     = jl.DataFrame(x_tr)
    dates_jl = _to_julia_dates(dates_series)
    param    = _build_param(cfg, H)
    data     = jl.HTBdata(y_train, x_jl, param, dates_jl)   # 4-arg, no fnames=

    jl.seval('import Logging; _htb_saved_logger = Logging.global_logger()')
    jl.seval('Logging.global_logger(Logging.SimpleLogger(devnull, Logging.Error))')
    try:
        output = jl.HTBfit(data, param)
    finally:
        jl.seval('Logging.global_logger(_htb_saved_logger)')

    yhat_tr = np.asarray(jl.HTBpredict(x_jl, output), dtype=float)
    yhat_te = None
    if x_test_df is not None:
        x_te    = prepare_x_v5(x_test_df)
        yhat_te = np.asarray(jl.HTBpredict(jl.DataFrame(x_te), output), dtype=float)

    importance = None
    if want_importance:
        try:
            rel   = jl.HTBrelevance(output, data, verbose=False)
            names = [str(n) for n in rel.fnames]
            fi    = np.asarray(rel.fi, dtype=float)
            importance = dict(zip(names, fi))
        except Exception as e:
            print(f'    [warn] HTBrelevance failed: {repr(e)[:60]}')

    tau_w, tau_table = None, None
    if want_tau:
        try:
            jl.seval('import Logging; _htb_saved_logger = Logging.global_logger()')
            jl.seval('Logging.global_logger(Logging.SimpleLogger(devnull, Logging.Error))')
            try:
                wt = jl.HTBweightedtau(output, data, verbose=False)
            finally:
                jl.seval('Logging.global_logger(_htb_saved_logger)')
            tau_w, tau_table = _extract_weightedtau(wt)   # pinned gavgtau scalar + df table
        except Exception as e:
            print(f'    [warn] HTBweightedtau extraction failed: {repr(e)[:120]}')
            tau_w, tau_table = None, None
    return output, yhat_tr, yhat_te, importance, tau_w, tau_table


print('Bridge helpers defined: fit_htboost_v5 (frozen-config threaded), prepare_x_v5')

### Probe — HTBweightedtau return schema (run once)
One-off diagnostic to read the installed build's `HTBweightedtau` field names. Run it, paste the `KEYS:`/`FIELD TYPES:` output back, then (if needed) pin `HTB_TAU_FIELD_OVERRIDES` in the bridge cell. Nothing depends on this cell.

In [ ]:
# ── PROBE (RUN ONCE, THEN PASTE THE OUTPUT BACK) ───────────────────────────────
# Discovers the installed HybridTreeBoosting build's HTBweightedtau RETURN SCHEMA so
# the τ^w field resolver (or HTB_TAU_FIELD_OVERRIDES, set in the bridge cell) can be
# confirmed against the REAL field names. This cell is SELF-CONTAINED and ISOLATED:
# nothing else in the notebook depends on it, and it only does work when you RUN it
# (it requires the juliacall HTBoost kernel). It fits one cheap model on SMOKE_SEC's
# richest fold, calls HTBweightedtau(verbose=True) (which also prints the τ table +
# the scalar "Average smoothing parameter τ is X"), then prints TYPE / KEYS /
# FIELD TYPES. Copy the KEYS: and FIELD TYPES: lines back so the names can be pinned.
def _probe_fit_one():
    _cfg = dict(lam=LAM, depth=None, nfold=NFOLD_INTERNAL,
                modality=(QUICK_MODALITY if QUICK_MODE else MODALITY), loss=LOSS)
    _panel = build_panel_v4(df_raw, [SMOKE_SEC], H)
    _fc = [c for c in _panel.columns if c not in META_COLS]
    _tr = _panel[_panel['date'] < pd.Timestamp(WF_FOLDS[-1][1])].copy()
    _x_tr, _, _ = fit_transform_xm_pca(_tr[_fc], _tr[_fc])
    _param = _build_param(_cfg, H)
    _data = jl.HTBdata(_tr['y'].to_numpy(float), jl.DataFrame(prepare_x_v5(_x_tr)),
                       _param, _to_julia_dates(_tr['date']))
    jl.seval('import Logging; _lg = Logging.global_logger()')
    jl.seval('Logging.global_logger(Logging.SimpleLogger(devnull, Logging.Error))')
    try:
        _out = jl.HTBfit(_data, _param)
    finally:
        jl.seval('Logging.global_logger(_lg)')
    return _out, _data


_probe_out, _probe_data = _probe_fit_one()        # live output + data from one fold
_tup = jl.HTBweightedtau(_probe_out, _probe_data, verbose=True)   # prints τ table + scalar
print("TYPE:", jl.seval('typeof')(_tup))
try:
    print("KEYS:", list(jl.seval('x -> collect(string.(keys(x)))')(_tup)))
except Exception as _e:
    print("not a keyed namedtuple:", repr(_e)[:80])
    try:
        print("PROPS:", list(jl.seval('x -> collect(string.(propertynames(x)))')(_tup)))
    except Exception as _e2:
        print("propertynames failed:", repr(_e2)[:80])
try:
    print("NFIELDS:", jl.seval('x -> length(x)')(_tup))
    print("FIELD TYPES:", jl.seval('x -> collect(string.(typeof.(values(x))))')(_tup))
except Exception as _e:
    print("len/values failed:", repr(_e)[:80])
print("\n[PROBE DONE] paste the KEYS:/PROPS: and FIELD TYPES: lines back; then pin "
      "HTB_TAU_FIELD_OVERRIDES in the bridge cell if auto-resolution needs help.")


### Feature taxonomy
Tag every feature column to a bucket (curve, momentum, vol, macro, credit, cross-market, Norway, carry/roll) for the feature-importance report and the macro-vs-curve comparison.

In [ ]:
# ── feature taxonomy — every column tagged to a bucket ──────────────────
# Buckets: curve, momentum, vol, macro, credit, cross_market, norway, carry_roll.
# Reused for feature-importance aggregation (report) and the macro-vs-curve
# comparison tied to Bianchi et al. The PCA components (xmpca_*) inherit the
# cross_market bucket, so importance attributed to compressed cross-market info is
# still bucketed correctly.

def bucket_feature(name):
    n = str(name)
    if n == 'carry_roll':
        return 'carry_roll'
    if n.startswith('xm_') or n.startswith('xmpca_'):
        return 'cross_market'
    if n.startswith('nor_') or n == 'swap_govt_spread':
        return 'norway'
    if n.startswith('ig_') or n.startswith('hy_'):
        return 'credit'
    if (n.startswith(('vol_', 'move_', 'vix_', 'v2x_', 'vxn_', 'ovx_', 'gvz_', 'rvx_'))):
        return 'vol'
    if n.startswith(('mom_', 'chg_lag')):
        return 'momentum'
    if (n in ('level_zscore', 'swap_ibor_basis') or
            n.startswith(('slope_', 'dev_ma_'))):
        return 'curve'
    # everything else: rates/macro/equity/commodity/breakeven exogenous → macro
    return 'macro'


# Build the taxonomy on the smoke-security panel feature columns (pre-PCA names;
# xm_* are listed individually and also summarised as their PCA collapse).
_tax_panel = build_panel_v4(df_raw, [SMOKE_SEC], H)
_tax_feats = [c for c in _tax_panel.columns if c not in META_COLS]
feature_taxonomy = pd.DataFrame({
    'feature': _tax_feats,
    'bucket':  [bucket_feature(c) for c in _tax_feats],
})
feature_taxonomy.to_csv(f'{OUT_DIR}/v5_feature_taxonomy.csv', index=False)

print('=== feature taxonomy ===')
_counts = feature_taxonomy['bucket'].value_counts().sort_index()
print(f'  {"bucket":<14s} {"#features (pre-PCA)":>20s}')
print('  ' + '-' * 36)
for b, c in _counts.items():
    print(f'  {b:<14s} {c:>20d}')
print(f'\n  Total feature columns (pre-PCA): {len(_tax_feats)}')
print(f'  cross_market (xm_*) collapse to ≤ {XM_PCA_KMAX} PCA comps in the fit path.')
print(f'  Saved: {OUT_DIR}/v5_feature_taxonomy.csv')

# Sanity: confirm Norway features are present & bucketed
_nor_feats = feature_taxonomy[feature_taxonomy['bucket'] == 'norway']['feature'].tolist()
print(f'\n  Norway-bucket features wired for {SMOKE_SEC}: {len(_nor_feats)}')
assert len(_nor_feats) > 0, 'No Norway features bucketed — check Part B wiring'
for c in _nor_feats[:40]:
    print(f'    {c}')

### Smoke test
A single end-to-end fit on the pilot security to verify the plumbing: the PCA round-trip, the Julia bridge, finite non-degenerate predictions, and a real train-vs-OOS gap. Magnitudes are reported, never tuned.

In [ ]:
# ── Smoke test (v5) — clean run with PCA compression + overfitting-surface report
# Trains SMOKE_SEC on the full expanded feature set, PCA-compresses xm_* on TRAIN
# rows only, fits, and scores BOTH train and OOS (Hiking fold). We REPORT the
# in-sample magnitude and the OOS-vs-0.50 distance; we never tune to move them.
# The harness intentionally limits in-sample fit (nfold=4 internal-CV early stopping
# + PCA compression of xm_* + accurate-modality regularisation), so this cell's
# tripwires test PLUMBING (bridge/PCA wired, above chance, real overfit gap, and a
# separate "can it overfit when unconstrained?" check) — not a fit magnitude.
assert SMOKE_SEC in df_raw.columns

SMOKE_CFG = dict(lam=LAM, depth=None, nfold=NFOLD_INTERNAL,
                 modality=(QUICK_MODALITY if QUICK_MODE else MODALITY), loss=LOSS)

_panel = build_panel_v4(df_raw, [SMOKE_SEC], H)
_fc    = [c for c in _panel.columns if c not in META_COLS]
_ts, _te_end = pd.Timestamp(WF_FOLDS[-1][1]), pd.Timestamp(WF_FOLDS[-1][2])
_tr = _panel[_panel['date'] < _ts].copy()
_te = _panel[(_panel['date'] >= _ts) & (_panel['date'] <= _te_end)].copy()
assert len(_tr) >= MIN_TRAIN_OBS and len(_te) >= MIN_TEST_OBS

# PCA fit on TRAIN rows only, applied to both
_x_tr, _x_te, _pca = fit_transform_xm_pca(_tr[_fc], _te[_fc])
_feat_count = _x_tr.shape[1]

# ── Step 1 diagnostics — verify it's not a real bug hiding behind "regularization"
# (a) every PCA component must be non-degenerate (constant component ⇒ real PCA bug)
_xmpca_cols = [c for c in _x_tr.columns if c.startswith('xmpca_')]
_xmpca_std  = _x_tr[_xmpca_cols].std()
print('PCA component std (train):',
      ' '.join(f'{s:.3g}' for s in _xmpca_std.values))
assert (_xmpca_std > 1e-9).all(), 'A xmpca_* component is constant — real PCA/bridge bug'
# (b) curve feature columns (level/slope/zscore) must survive into the model matrix
_curve_cols = [c for c in _x_tr.columns if bucket_feature(c) == 'curve']
assert 'level_zscore' in _x_tr.columns, 'level_zscore missing from model matrix — real bug'
assert any(c.startswith('slope_') for c in _x_tr.columns), 'no slope_* cols — real bug'
assert len(_curve_cols) >= 3, f'too few curve cols ({len(_curve_cols)}) — real bug'
# (c) spot-check the bucket map on the ACTUAL model-input columns (cosmetic: report only)
_bucket_examples = {}
for c in _x_tr.columns:
    _bucket_examples.setdefault(bucket_feature(c), []).append(c)
print('Bucket spot-check (model-matrix columns; affects the importance report only):')
for b in sorted(_bucket_examples):
    ex = _bucket_examples[b][:5]
    print(f'  {b:<14s} n={len(_bucket_examples[b]):3d}  e.g. {", ".join(ex)}')

print(f'\nSmoke test: {SMOKE_SEC}   (config={SMOKE_CFG})')
print(f'  Features pre-PCA: {len(_fc)}  (xm_*={sum(1 for f in _fc if f.startswith("xm_"))})')
print(f'  PCA applied={_pca["applied"]}  k={_pca["k"]}  '
      f'(target ≥{_pca["var_target"]:.0%} train var → cum={_pca["evr_cum_k"]:.3f})')
print(f'  Features post-PCA: {_feat_count}   Train n={len(_tr)}  Test n={len(_te)}')

t0 = time.time()
_out, _yhat_tr, _yhat_te, _imp, _tau_w, _tau_tab = fit_htboost_v5(
    _tr['y'].to_numpy(float), _x_tr, _tr['date'], _x_te, SMOKE_CFG, H,
    want_importance=True, want_tau=True)
if _tau_w is not None:
    print(f'  Smoke τ^w (importance-weighted smoothness): {_tau_w:.2f}  '
          f'→ band={tau_w_band(_tau_w)}  '
          f'(per-feature table rows={0 if _tau_tab is None else len(_tau_tab)})')
print(f'  Fit time: {time.time()-t0:.1f}s')

_s_tr = _score(_tr['y'].to_numpy(float), _yhat_tr)
_s_te = _score(_te['y'].to_numpy(float), _yhat_te)
_n_eff = max(1, round(_s_te['n_obs'] / max(H, 1)))
_se    = 0.5 / np.sqrt(_n_eff)
print(f'\n  Train:  DirAcc={_s_tr["dir_acc"]:.3f}  R²={_s_tr["r2"]:+.4f}  n={_s_tr["n_obs"]}')
print(f'  OOS:    DirAcc={_s_te["dir_acc"]:.3f}  R²={_s_te["r2"]:+.4f}  n={_s_te["n_obs"]}  '
      f'(n_eff≈{_n_eff}, 1 SE≈{_se:.3f})')
print(f'  OOS distance from 0.50: {abs(_s_te["dir_acc"]-0.5)/_se:.2f} SE  '
      f'(expected ≈ 0; we REPORT, never tune)')

# ── Overfitting-surface report ─────────────────────────────────────────
print('\n=== Overfitting-surface report (smoke) ===')
print(f'  features pre-PCA (uncompressed): {len(_fc)}')
print(f'  v5 feature count (post-PCA):    {_feat_count}   '
      f'(Δ = {len(_fc) - _feat_count} fewer)')
print(f'  PCA k / explained var:          {_pca["k"]} comps / {_pca["evr_cum_k"]:.3f}')
# Step 2 — methodological note: the kmax cap may bind below the variance target.
_cap_binds = (_pca['k'] >= XM_PCA_KMAX) and (_pca['evr_cum_k'] < _pca['var_target'] - 1e-9)
print(f'  PCA cap XM_PCA_KMAX={XM_PCA_KMAX} binds: {_cap_binds}   '
      f'(cross-market variance dropped ≈ {(1 - _pca["evr_cum_k"])*100:.0f}%)')
if _cap_binds:
    print(f'    NOTE (methods): the {XM_PCA_KMAX}-component cap is reached before the '
          f'{_pca["var_target"]:.0%} target, so cross-market info is heavily compressed.')
    print(f'    Raising XM_PCA_KMAX is an a-priori decision that must be applied '
          f'uniformly to every tenor/horizon and recorded in the methods. '
          f'Default: keep {XM_PCA_KMAX} and document the compression.')
print(f'  Provisional reg config:         {SMOKE_CFG}')
print(f'  (frozen reg config is selected by inner-CV on TRAIN in the next cell)')
if _imp:
    _imp_b = {}
    for k, v in _imp.items():
        _imp_b[bucket_feature(k)] = _imp_b.get(bucket_feature(k), 0.0) + v
    print('\n  Smoke feature importance by bucket (train fit, sums to ~100):')
    for b, v in sorted(_imp_b.items(), key=lambda kv: -kv[1]):
        print(f'    {b:<14s} {v:6.1f}')

# Plumbing only: predictions exist, are finite, and are not degenerate. We do NOT
# assert an in-sample magnitude — modality='accurate' tunes for OOS via internal
# block-CV and is designed NOT to overfit train, so a large in-sample fit is not
# expected. Train/OOS are REPORTED above for information.
assert np.isfinite(_yhat_te).all(), 'NaN/Inf in OOS preds — stale workers?'
assert np.nanstd(_yhat_te) > 1e-9, 'OOS predictions all identical — fit failed'
assert np.nanstd(_yhat_tr) > 1e-9, 'Train predictions all identical — fit failed'
print(f'\n[PASS] Smoke v5 — clean run, PCA round-trip, populated OOS')
print(f'  train DirAcc={_s_tr["dir_acc"]:.3f}, OOS DirAcc={_s_te["dir_acc"]:.3f} '
      f'(reported, not asserted)')
print('  modality=accurate optimizes OOS via internal block-CV; in-sample fit is')
print('  not expected to be large — this is intended behaviour, not a failure.')

### Freeze the regularization
Pre-commit the L2 `lambda` to the midpoint of the grid and build the frozen config reused for every horizon, tenor and fold. Fixing it a priori keeps out-of-sample results from influencing model selection.

In [ ]:
# ── L2 regularization fixed A PRIORI (no data-driven selection) ──────────
# HTBcv is broken in the installed HybridTreeBoosting build (UndefVarError: `modality`
# inside HTBcv). More importantly, for a null-confirmation harness we do NOT tune
# lambda. We PRE-COMMIT lambda to the midpoint of LAM_GRID, applied uniformly to every
# horizon, tenor, fold, and the block-CV path. Robustness across lambda is shown by the
# λ-sensitivity cell, which REPORTS the null at each lambda and never re-selects from OOS.
LAM_FIXED = float(LAM_GRID[len(LAM_GRID) // 2])        # pre-committed midpoint (= 0.10)

FROZEN_CONFIG = dict(lam=LAM_FIXED, depth=None, nfold=NFOLD_INTERNAL,
                     modality=(QUICK_MODALITY if QUICK_MODE else MODALITY), loss=LOSS)
LAM         = LAM_FIXED
FROZEN_HASH = config_hash(FROZEN_CONFIG,
                          extra=f'pca_var{XM_PCA_VAR}_kmax{XM_PCA_KMAX}_tl{TARGET_LAGS}')

print('=== regularization: lambda fixed a priori (no tuning) ===')
print(f'  lambda = {LAM_FIXED}  (pre-committed midpoint of {LAM_GRID})')
print(f'  FROZEN_CONFIG = {FROZEN_CONFIG}')
print(f'  config_hash   = {FROZEN_HASH}')
print('  [PASS] lambda is a-priori and uniform — no OOS/regime fold influences it.')

### Walk-forward scorer
The shared fit-and-score routine: PCA-compress on training rows, fit, then emit one train row and one OOS row per fold using the common metrics function. Reused by both the walk-forward and block-CV paths.

In [ ]:
# ── shared fit→(train+oos) scorer, reused by walk-forward AND block-CV ──
FEAT_SPEC = f'pca_var{XM_PCA_VAR}_kmax{XM_PCA_KMAX}_tl{TARGET_LAGS}'


def _meta_base(sec, H, cfg, scheme, fold, regime, feature_count,
               target_kind='level_change'):
    country, tenor = sec.rsplit('_', 1)
    return dict(validation_scheme=scheme, target_kind=target_kind,
                security=sec, country=country, tenor=tenor, fold=fold, regime=regime,
                config_hash=config_hash(cfg, extra=FEAT_SPEC), feature_count=feature_count)


def _fit_score_rows(tr, te, fc, H, cfg, sec, scheme, fold, regime,
                    want_importance=False, want_tau=False, target_kind='level_change'):
    """PCA-compress (fit on tr only) → fit → emit a TRAIN row and an OOS row.
    Returns (rows, importance_dict_or_None, tau_record_or_None). The τ record is a
    model DIAGNOSTIC (smoothness) carried separately from the shared metrics rows."""
    x_tr, x_te, pca = fit_transform_xm_pca(tr[fc], te[fc])
    feat_count = x_tr.shape[1]
    out, yhat_tr, yhat_te, imp, tau_w, tau_table = fit_htboost_v5(
        tr['y'].to_numpy(float), x_tr, tr['date'], x_te, cfg, H,
        want_importance=want_importance, want_tau=want_tau)
    # (b) data-chosen complexity read from the fitted output (read-only -> forecast-neutral)
    try:
        _ntrees = int(jl.seval('o -> Int(o.ntrees)')(out))
        _depth  = int(jl.seval('o -> Int(o.bestparam.depth)')(out))
    except Exception:
        _ntrees, _depth = -1, -1
    base = _meta_base(sec, H, cfg, scheme, fold, regime, feat_count, target_kind)
    row_tr = compute_metrics_row(tr['y'].to_numpy(float), yhat_tr, H,
                                 {**base, 'sample': 'train'})
    row_te = compute_metrics_row(te['y'].to_numpy(float), yhat_te, H,
                                 {**base, 'sample': 'oos'})
    row_tr['pca_k'] = pca['k']; row_te['pca_k'] = pca['k']
    # Log the REALISED cumulative explained-variance retained by the k components.
    # The 12-component cap typically binds before the 95% target, so this is ≈0.70,
    # not 0.95 — recorded per fold so the true figure is logged, never assumed.
    # (Extra column, exactly like pca_k; the shared htb_metrics schema is untouched.)
    row_tr['xm_pca_evr'] = pca['evr_cum_k']; row_te['xm_pca_evr'] = pca['evr_cum_k']
    row_tr['htb_ntrees'] = _ntrees; row_te['htb_ntrees'] = _ntrees
    row_tr['htb_depth']  = _depth;  row_te['htb_depth']  = _depth
    # per-obs OOS predictions (forecast-neutral: packages the already-computed yhat_te)
    _country, _tenor = sec.rsplit('_', 1)
    preds = pd.DataFrame({'date': te['date'].values, 'security': sec, 'tenor': _tenor,
                          'horizon': int(H), 'regime': regime, 'scheme': scheme,
                          'y_true': te['y'].to_numpy(float),
                          'y_pred': np.asarray(yhat_te, float)})
    tau_rec = None
    if want_tau and tau_w is not None:
        tau_rec = {'tau_w': float(tau_w), 'tau_w_band': tau_w_band(tau_w),
                   'table': tau_table}
    return [row_tr, row_te], imp, tau_rec, preds


def run_security_v5(sec, df_raw, H, cfg, want_importance=False, want_tau=False,
                    verbose=False):
    """Walk-forward (causal, expanding window) across all folds for one security at
    horizon H. Emits one TRAIN row and one OOS row per populated fold. One-sided
    pre-test OVERLAP purge (training labels reaching into the test window)."""
    if sec not in df_raw.columns:
        return [], [], [], []
    panel = build_panel_v4(df_raw, [sec], H)
    if len(panel) == 0:
        return [], [], [], []
    fc = [c for c in panel.columns if c not in META_COLS]
    rows, imps, taus, preds_all = [], [], [], []
    for fold_name, test_start, test_end, regime in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(test_start), pd.Timestamp(test_end)
        purge_ts = ts_ts - pd.tseries.offsets.BDay(H - 1)          # OVERLAP = H-1
        tr = panel[panel['date'] < purge_ts].copy()
        te = panel[(panel['date'] >= ts_ts) & (panel['date'] <= te_ts)].copy()
        if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
            if verbose:
                print(f'  {fold_name}: skip (train={len(tr)}, test={len(te)})')
            continue
        try:
            r, imp, tau, preds = _fit_score_rows(tr, te, fc, H, cfg, sec, 'walk_forward',
                                          fold_name, regime,
                                          want_importance=want_importance, want_tau=want_tau)
        except Exception as e:
            if verbose:
                print(f'  {fold_name}: FAILED — {repr(e)[:80]}')
            continue
        rows.extend(r)
        preds_all.append(preds)
        if imp is not None:
            imps.append({'security': sec, 'horizon': H, 'fold': fold_name, 'imp': imp})
        if tau is not None:
            _country, _tenor = sec.rsplit('_', 1)
            taus.append({'security': sec, 'country': _country, 'tenor': _tenor,
                         'horizon': H, 'fold': fold_name, 'regime': regime,
                         'tau_w': tau['tau_w'], 'tau_w_band': tau['tau_w_band'],
                         'table': tau['table']})
        if verbose:
            oos = r[1]
            print(f'  {fold_name:<14s} train DA={r[0]["dir_acc"]:.3f}  '
                  f'OOS DA={oos["dir_acc"]:.3f}  CT-R²={oos["ct_r2_oos"]:+.4f}  n={oos["n_obs"]}')
    return rows, imps, taus, preds_all


print('Walk-forward runner defined: run_security_v5 (emits train+oos rows per fold)')

### Resolve the horizon grid
Use the standard horizons and add the long horizons (6m/12m) only when the data supports at least one valid fold; otherwise they are logged and skipped.

In [ ]:
# ── resolve the horizon grid; gate 126/252 on data length ───────────────
# H_GRID always runs. Long horizons (126/252) are included ONLY if the data length
# supports at least one walk-forward fold with MIN_TRAIN_OBS / MIN_TEST_OBS for the
# smoke security; otherwise they are logged and skipped (never silently dropped).

def _horizon_supported(h):
    panel = build_panel_v4(df_raw, [SMOKE_SEC], h)
    if len(panel) == 0:
        return False
    for _, ts, te, _r in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(ts), pd.Timestamp(te)
        purge = ts_ts - pd.tseries.offsets.BDay(h - 1)
        ntr = (panel['date'] < purge).sum()
        nte = ((panel['date'] >= ts_ts) & (panel['date'] <= te_ts)).sum()
        if ntr >= MIN_TRAIN_OBS and nte >= MIN_TEST_OBS:
            return True
    return False


if QUICK_MODE:
    HORIZONS = list(QUICK_HGRID)
    SWEEP_TENORS = [s for s in QUICK_TENORS if s in UNIVERSE]
    print(f'[QUICK_MODE] HORIZONS={HORIZONS}  SWEEP_TENORS={SWEEP_TENORS}')
else:
    HORIZONS = list(H_GRID)
    for h in H_GRID_LONG:
        if _horizon_supported(h):
            HORIZONS.append(h)
            print(f'  long horizon H={h}: data supports it → included')
        else:
            print(f'  long horizon H={h}: insufficient data per fold → SKIPPED (logged)')
    SWEEP_TENORS = list(UNIVERSE)

HORIZONS = sorted(set(HORIZONS))
print(f'\nFinal horizon grid: {HORIZONS}')
print(f'Sweep tenors ({len(SWEEP_TENORS)}): {SWEEP_TENORS}')
print(f'Frozen config: {FROZEN_CONFIG}')
assert FROZEN_CONFIG is not None, 'Run the inner-CV selection cell first'

### Walk-forward sweep
The main causal run: the frozen model across every horizon, tenor and fold, emitting train and OOS rows. This is heavy (minutes per fit); gate it with the run switches. Partial results are written for inspection.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# REPRO_HANDSHAKE — paste IDENTICALLY at the top of BOTH GB notebooks, AFTER the
# data load + fit-function defs + (pooled) XM_BLOCK, but BEFORE the heavy sweep.
# Part A prints a deterministic FINGERPRINT to eyeball-diff across machines.
# Part B runs a ~1-min pilot fit through the REAL fit→score→persist path.
# ═══════════════════════════════════════════════════════════════════════════════
import os as _os, json as _json, hashlib as _hashlib, pickle as _pickle
import numpy as _np, pandas as _pd
import src.config as _C                       # read config WITHOUT shadowing notebook globals
from src.evaluation.metrics import SHARED_COLS as _SHARED

def _sha16(b):                                 # stable, unsalted SHA-256 (first 16 hex)
    return _hashlib.sha256(b).hexdigest()[:16]

def _hash_df(df):
    return _sha16(_pd.util.hash_pandas_object(df, index=True).values.tobytes())

def _resolve(*names):                          # first present, non-None global
    g = globals()
    for n in names:
        if n in g and g[n] is not None:
            return n, g[n]
    return None, None

# ── Part A — deterministic fingerprint ─────────────────────────────────────────
_L = ['FINGERPRINT:']
if 'df_raw' in globals():
    _dfr = globals()['df_raw']
    # DATA-INTEGRITY GATE: raw (pre-augmentation) values-only hash, portable across CPU arch
    # (round_trip float parse + deterministic column order). df_raw.hash16 below is INFORMATIONAL:
    # the ~88 derived columns differ at ULP (~1e-15) across x64/arm64 SIMD, which does not move results.
    try:
        from src.data.bloomberg import load_data as _load_data
        _raw_vfp = _sha16(_pd.util.hash_pandas_object(_load_data(), index=False).values.tobytes())
    except Exception as _e:
        _raw_vfp = f'(unavailable: {repr(_e)[:40]})'
    _L += [f'  df_raw.shape         = {tuple(_dfr.shape)}',
           f'  df_raw_values.hash16 = {_raw_vfp}   [DATA GATE - must match]',
           f'  df_raw.hash16        = {_hash_df(_dfr)}   [informational - ULP arch-variant in derived cols]',
           f'  df_raw.index_min     = {_dfr.index.min()}',
           f'  df_raw.index_max     = {_dfr.index.max()}']
else:
    _L.append('  df_raw              = (unavailable — load data before the handshake)')

_pn_name, _pn = _resolve('panel', 'PANEL', 'pooled_panel', 'df_panel')
if isinstance(_pn, _pd.DataFrame):
    _L += [f'  panel({_pn_name}).shape  = {tuple(_pn.shape)}',
           f'  panel.hash16        = {_hash_df(_pn)}']
else:
    _L.append('  panel               = (not built at handshake point — informational)')

_cfg = {'JULIA_SEED': _C.JULIA_SEED, 'H_GRID': _C.H_GRID, 'WF_FOLDS': _C.WF_FOLDS,
        'BLOCK_CV_FOLDS': _C.BLOCK_CV_FOLDS, 'NOR_TENORS': _C.NOR_TENORS,
        'MIN_TRAIN_OBS': _C.MIN_TRAIN_OBS, 'MIN_TEST_OBS': _C.MIN_TEST_OBS,
        'ALPHA_MT': _C.ALPHA_MT, 'XM_PCA_ENABLE': _C.XM_PCA_ENABLE,
        'XM_PCA_VAR': _C.XM_PCA_VAR, 'XM_PCA_KMAX': _C.XM_PCA_KMAX,
        'FROZEN_CONFIG': globals().get('FROZEN_CONFIG')}
_L.append(f'  config.hash16       = {_sha16(_json.dumps(_cfg, sort_keys=True, default=str).encode())}')

_meta = globals().get('META_COLS', {'date', 'security', 'y', 'level'})
_feat = None
if isinstance(_pn, _pd.DataFrame):
    _feat = sorted(c for c in _pn.columns if c not in _meta)
else:
    for _nm in ('NOR_FEATURE_COLS', 'FEATURE_COLS', 'feature_cols', 'feat_cols'):
        if globals().get(_nm):
            _feat = sorted(globals()[_nm]); break
if _feat is not None:
    _L += [f'  feature_spec.n      = {len(_feat)}',
           f'  feature_spec.hash16 = {_sha16(_json.dumps(_feat, sort_keys=True).encode())}']
else:
    _L.append('  feature_spec        = (unavailable at handshake point)')

import numpy as _vnp, pandas as _vpd, scipy as _vsp
_L += [f'  MACHINE_ID          = {_C.MACHINE_ID}',
       f'  versions            = numpy {_vnp.__version__} / pandas {_vpd.__version__} / scipy {_vsp.__version__}']
_jl = globals().get('jl')
try:
    if _jl is not None:
        _jv = str(_jl.seval('string(VERSION)'))
        try:
            _hv = str(_jl.seval('string(pkgversion(HybridTreeBoosting))'))
        except Exception:
            _hv = '?'
        _L.append(f'  julia / HTBoost     = julia {_jv} / HybridTreeBoosting {_hv}')
    else:
        _L.append('  julia / HTBoost     = (jl not loaded)')
except Exception as _e:
    _L.append(f'  julia / HTBoost     = (unreachable: {repr(_e)[:50]})')
print('\n'.join(_L))
print('  ⇒ eyeball-diff df_raw / config / feature_spec (and panel) hashes vs your partner')
print('    BEFORE the full sweep; any mismatch ⇒ NOT the same inputs ⇒ stop and reconcile.\n')

# ── Part B — pilot fit (~8 min) through the REAL fit→score→persist path ─────────
# EXACTLY ONE fit (1 security × 1 horizon × 1 fold) at the REAL FROZEN_CONFIG — NO
# modality / nfold / nofullsample override — so it is byte-path-identical to a sweep
# fit, just run once. Cheap by SCOPE, not by modality: τ^w (gavgtau) is only the
# reported quantity when HTBoost actually tunes (modality ∈ {compromise, accurate});
# a fast/fastest fit would emit a DIFFERENT, untuned τ^w and give false cross-machine
# confidence. This one-time ~8-min per-machine cost is the point — it exercises the
# tuning code path where any cross-machine FP divergence would surface. Pilot outputs
# go to results/<OUT_DIR>/_pilot/... (gitignored).
import time as _time
_TAG       = globals().get('NOTEBOOK_TAG', 'nb')
_PILOT_DIR = f'{OUT_DIR}/_pilot/{_TAG}__{_C.MACHINE_ID}'
_os.makedirs(_PILOT_DIR, exist_ok=True)
_SEC, _H   = 'NOR_10Y', 21

def _need(field, ok):
    assert ok, f'PILOT FAIL: missing/empty output field -> {field}'

_cfg_pilot = globals().get('FROZEN_CONFIG')    # the REAL sweep config; do NOT override
_need('FROZEN_CONFIG set (run the regularization cell first)',
      isinstance(_cfg_pilot, dict) and len(_cfg_pilot) > 0)
_modality  = _cfg_pilot.get('modality') or globals().get('MODALITY')
_FOLD      = WF_FOLDS[-1]                       # last (Hiking, expanding) fold → most training data
_t0        = _time.time()

_rows, _imp_present, _tau_w, _sidecars = [], False, None, []
if 'run_pooled_wf' in globals():               # ── POOLED notebook ──
    _pcsv = f'{_PILOT_DIR}/pilot_pooled_wf_H{_H}.csv'
    _ppkl = f'{_PILOT_DIR}/pilot_pooled_wf_imps_H{_H}.pkl'
    for _f in (_pcsv, _ppkl):
        if _os.path.exists(_f):
            _os.remove(_f)
    _ck = _make_checkpoint(_pcsv, _ppkl)
    _saved_wf = list(WF_FOLDS)
    try:
        globals()['WF_FOLDS'] = [_FOLD]
        _rows, _imps = run_pooled_wf(df_raw, [_SEC], _H, _cfg_pilot,
                                     use_categoricals=False, want_importance=True,
                                     verbose=False, xm_block=globals().get('XM_BLOCK'),
                                     checkpoint=_ck)
    finally:
        globals()['WF_FOLDS'] = _saved_wf
    _oos = [r for r in _rows if r.get('sample') == 'oos']
    _imp_present = len(_imps) > 0
    _tau_w = (_oos[0].get('tau_w') if _oos else None)
    _sidecars = [_pcsv]
elif 'run_security_v5' in globals():           # ── PER-SECURITY notebook ──
    _saved_wf = list(WF_FOLDS)
    try:
        globals()['WF_FOLDS'] = [_FOLD]
        _rows, _imps, _taus, _ = run_security_v5(_SEC, df_raw, _H, _cfg_pilot,
                                              want_importance=True, want_tau=True, verbose=False)
    finally:
        globals()['WF_FOLDS'] = _saved_wf
    _oos = [r for r in _rows if r.get('sample') == 'oos']
    _pcsv = f'{_PILOT_DIR}/pilot_metrics_H{_H}.csv'
    _pd.DataFrame(_rows).to_csv(_pcsv, index=False)
    if _imps:
        with open(f'{_PILOT_DIR}/pilot_imps_H{_H}.pkl', 'wb') as _fh:
            _pickle.dump(_imps, _fh)
    if _taus:
        _pd.DataFrame([{k: v for k, v in t.items() if k != 'table'} for t in _taus]
                      ).to_csv(f'{_PILOT_DIR}/pilot_tau_H{_H}.csv', index=False)
    _imp_present = len(_imps) > 0
    _tau_w = (_taus[0].get('tau_w') if _taus else None)
    _sidecars = [_pcsv]
else:
    raise RuntimeError('PILOT FAIL: neither run_pooled_wf nor run_security_v5 in scope — '
                       'paste the handshake AFTER the fit-function cells.')

assert len(WF_FOLDS) == 5, 'PILOT FAIL: WF_FOLDS not restored to 5 folds after the pilot slice'
_mins = (_time.time() - _t0) / 60.0

# ── assertions: every expected field present + non-empty; fail LOUDLY by name ──
_need('pilot produced rows', len(_rows) > 0)
_need('oos row', len(_oos) > 0)
_row = _oos[0]
for _c in _SHARED:                             # schema completeness: every shared field present
    _need(f'SHARED_COLS[{_c}]', (_c in _row) and (_row[_c] is not None))
for _c in ('n_obs', 'dir_acc', 'r2_raw', 'mse_model', 'mse_rw', 'ct_r2_oos', 'binom_pval'):
    _v = _row.get(_c)                          # core metrics must be real (not NaN)
    _need(f'{_c} (NaN)', _v is not None and not (isinstance(_v, float) and _np.isnan(_v)))
_need('n_obs > 0', int(_row['n_obs']) > 0)
_need('tau_w', _tau_w is not None and not (isinstance(_tau_w, float) and _np.isnan(_tau_w)))
_need('importance record', _imp_present)
for _sc in _sidecars:
    _need(f'sidecar CSV {_sc}', _os.path.exists(_sc) and _os.path.getsize(_sc) > 0)

print(f'PILOT OK (modality={_modality}, ~{_mins:.0f}m): all output fields present  '
      f'({_SEC} H={_H} fold={_FOLD[0]}; rows={len(_rows)}; tau_w={_tau_w:.3f}; '
      f'sidecars={[_os.path.basename(s) for s in _sidecars]}; under {_PILOT_DIR}/)')


In [ ]:
# -- Per-security WF checkpoint (Prompt 5b): machine-stamped, fold-batch append, resume --
# Mirrors the pooled _make_checkpoint/_done_keys_from_csv, per-security (no cat arm). run_security_v5
# is ATOMIC over folds (no per-fold hook), so resume is (security, horizon)-granular: each completed
# security's full fold-batch is appended (flush+fsync) immediately and SKIPPED on resume if already on
# disk. The fit (run_security_v5/_fit_score_rows/JULIA_SEED/FROZEN_CONFIG) is untouched -- this only
# adds the surrounding persistence + the cross-machine tau CSV sidecar + resume pickles for tau/imps.
import pickle as _pkl_v5
from src.config import MACHINE_ID as _MID_v5
from src.utils.gitpush import push_results as _push_results_v5

V5_WF_CSV  = lambda h: f'{OUT_DIR}/v5_wf_H{h}__{_MID_v5}.csv'
V5_TAU_CSV = lambda h: f'{OUT_DIR}/v5_tau_H{h}__{_MID_v5}.csv'
V5_TAU_PKL = lambda h: f'{OUT_DIR}/v5_tau_H{h}__{_MID_v5}.pkl'
V5_IMP_PKL = lambda h: f'{OUT_DIR}/v5_wf_imps_H{h}__{_MID_v5}.pkl'
V5_PRED_CSV = lambda h: f'{OUT_DIR}/v5_wf_preds_H{h}__{_MID_v5}.csv'
V5_IMP_CSV = lambda h: f'{OUT_DIR}/v5_wf_imps_H{h}__{_MID_v5}.csv'


def _v5_done_secs(csv_path):
    """Securities already fully scored in csv_path (run_security_v5 is atomic over folds)."""
    if not os.path.exists(csv_path):
        return set()
    try:
        return set(pd.read_csv(csv_path, usecols=['security'])['security'].astype(str))
    except Exception as e:
        print(f'  [resume] could not read {csv_path} ({repr(e)[:50]}) -- treating as empty')
        return set()


def _v5_load_pkl(path):
    if not os.path.exists(path):
        return []
    try:
        with open(path, 'rb') as f:
            return list(_pkl_v5.load(f))
    except Exception:
        return []


def _v5_make_checkpoint(h):
    """Checkpoint for horizon h: 'done' security set seeded from disk + append(rows, taus, imps) that
    durably writes one security's fold-batch (CSV append, flush+fsync), re-dumps the tau/imps pickles,
    and refreshes the tau CSV sidecar (without the per-fold 'table' object)."""
    csv_p, tau_csv_p, tau_pkl_p, imp_p, pred_p, imp_csv_p = V5_WF_CSV(h), V5_TAU_CSV(h), V5_TAU_PKL(h), V5_IMP_PKL(h), V5_PRED_CSV(h), V5_IMP_CSV(h)
    state = {'done': _v5_done_secs(csv_p),
             'taus': _v5_load_pkl(tau_pkl_p),
             'imps': _v5_load_pkl(imp_p),
             'cols': list(pd.read_csv(csv_p, nrows=0).columns) if os.path.exists(csv_p) else None,
             'pcols': list(pd.read_csv(pred_p, nrows=0).columns) if os.path.exists(pred_p) else None}

    def _append(rows, taus, imps, preds=None):
        if rows:
            if state['cols'] is None:
                state['cols'] = list(pd.DataFrame(rows).columns)
            d = pd.DataFrame(rows).reindex(columns=state['cols'])
            write_header = not os.path.exists(csv_p)
            with open(csv_p, 'a', newline='') as fh:
                d.to_csv(fh, header=write_header, index=False); fh.flush(); os.fsync(fh.fileno())
        if taus:
            state['taus'].extend(taus)
            with open(tau_pkl_p, 'wb') as fh:
                _pkl_v5.dump(state['taus'], fh); fh.flush(); os.fsync(fh.fileno())
            pd.DataFrame([{k: v for k, v in t.items() if k != 'table'} for t in state['taus']]
                         ).to_csv(tau_csv_p, index=False)
        if imps:
            state['imps'].extend(imps)
            with open(imp_p, 'wb') as fh:
                _pkl_v5.dump(state['imps'], fh); fh.flush(); os.fsync(fh.fileno())
            pd.DataFrame([{'security': _r['security'], 'horizon': _r['horizon'], 'fold': _r['fold'],
                           'feature': _f, 'relevance': _v}
                          for _r in state['imps'] for _f, _v in (_r['imp'] or {}).items()]
                         ).to_csv(imp_csv_p, index=False)
        if preds is not None and len(preds):
            if state['pcols'] is None:
                state['pcols'] = list(preds.columns)
            dp = preds.reindex(columns=state['pcols'])
            write_header = not os.path.exists(pred_p)
            with open(pred_p, 'a', newline='') as fh:
                dp.to_csv(fh, header=write_header, index=False); fh.flush(); os.fsync(fh.fileno())

    return {'done': state['done'], 'append': _append}


print('Per-security WF checkpoint defined: V5_WF_CSV/V5_TAU_CSV(h), _v5_make_checkpoint(h) '
      '[(security, horizon)-granular resume; fit untouched]')

In [ ]:
# -- WALK-FORWARD SWEEP -- H_GRID x tenors x folds, train+OOS rows --
# Uses the FROZEN config for every horizon and tenor (never re-selected). This is the heavy cell.
# WARNING: with MODALITY='accurate' each fit is minutes; the full grid is hours -- gate with
# RUN_WALK_FORWARD / QUICK_MODE. Prompt 5b: each completed (security, horizon) fold-batch is APPENDED
# to a machine-stamped per-horizon CSV (flush+fsync) the moment it finishes, and SKIPPED on resume if
# already on disk; df_wf / wf_taus / wf_imps are reassembled from the stamped files so a resumed run is
# complete. The run_security_v5 call below is byte-identical to before -- only persistence is added.
wf_rows, wf_imps, wf_taus = [], [], []
_wf_failed = []

if not RUN_WALK_FORWARD:
    print('[skipped] RUN_WALK_FORWARD = False')
else:
    _t0 = time.time()
    _total = len(HORIZONS) * len(SWEEP_TENORS)
    _i = 0
    for H_run in HORIZONS:
        _ck = _v5_make_checkpoint(H_run)          # resume: skip securities already on disk
        if _ck['done']:
            print(f'[WF H={H_run}] resume: {len(_ck["done"])} security(ies) already on disk -> skip them')
        for sec in SWEEP_TENORS:
            _i += 1; _ts = time.time()
            if sec in _ck['done']:
                continue
            try:
                rows, imps, taus, preds = run_security_v5(sec, df_raw, H_run, FROZEN_CONFIG,
                                                   want_importance=True, want_tau=True,
                                                   verbose=False)
            except Exception as e:
                print(f'  [{_i}/{_total}] H={H_run} {sec}: FAILED {repr(e)[:70]}')
                _wf_failed.append(('walk_forward', H_run, sec))
                continue
            _ck['append'](rows, taus, imps, pd.concat(preds, ignore_index=True) if preds else None)  # durable append (flush+fsync) per completed (sec, H)
            wf_rows.extend(rows); wf_imps.extend(imps); wf_taus.extend(taus)
            _oos = [r for r in rows if r['sample'] == 'oos']
            _da = ', '.join(f'{r["dir_acc"]:.3f}' for r in _oos)
            print(f'  [{_i}/{_total}] H={H_run:<3d} {sec:<10s} folds={len(_oos)} '
                  f'OOS_DA=[{_da}]  ({time.time()-_ts:.1f}s)')
        if config.AUTO_PUSH:
            _push_results_v5([V5_WF_CSV(H_run), V5_TAU_CSV(H_run), V5_PRED_CSV(H_run), V5_IMP_CSV(H_run)],
                             f'results: v5_nor walk_forward H{H_run} @ {_MID_v5}')
    print(f'\nWalk-forward sweep done in {(time.time()-_t0)/60:.1f} min')
    if _wf_failed:
        print(f'  {len(_wf_failed)} cells failed: {_wf_failed}')

# Reassemble from this machine's stamped per-horizon files (complete across resumed sessions).
if RUN_WALK_FORWARD:
    _wf_parts = [pd.read_csv(V5_WF_CSV(h)) for h in HORIZONS if os.path.exists(V5_WF_CSV(h))]
    df_wf = pd.concat(_wf_parts, ignore_index=True) if _wf_parts else pd.DataFrame(wf_rows)
    wf_taus = [t for h in HORIZONS for t in _v5_load_pkl(V5_TAU_PKL(h))]
    wf_imps = [im for h in HORIZONS for im in _v5_load_pkl(V5_IMP_PKL(h))]
else:
    df_wf = pd.DataFrame(wf_rows)
print(f'df_wf shape: {df_wf.shape}')

### Lambda-sensitivity check
Re-run the walk-forward at each lambda in the grid to show the result is insensitive to L2 strength. A robustness appendix only — it never re-selects lambda and is excluded from the primary metrics and the multiple-testing families.

In [ ]:
# ── λ-sensitivity robustness (REPORT ONLY — never a selection) ──────────
# Re-runs the causal walk-forward at every lambda in LAM_GRID to show the null is
# insensitive to L2 strength. This is a robustness appendix: it NEVER changes
# FROZEN_CONFIG / LAM, never picks a best-OOS lambda, and never writes into
# v5_metrics_long.csv or either MTC family. Scope kept cheap: SMOKE_SEC, primary
# horizon H, walk-forward only. Per-fold metrics reuse compute_metrics_row (via
# run_security_v5) so numbers match the main run exactly.
if not RUN_WALK_FORWARD:
    print('[λ-sensitivity skipped] RUN_WALK_FORWARD = False')
    lambda_sens_summary = pd.DataFrame()
else:
    _ls_rows, _ls_summary = [], []
    print(f'=== λ-sensitivity (report only): {SMOKE_SEC}, H={H}, walk-forward ===')
    for _lam_s in LAM_GRID:
        _cfg_s = {**FROZEN_CONFIG, 'lam': float(_lam_s)}
        try:
            _rows_s, _, _, _ = run_security_v5(SMOKE_SEC, df_raw, H, _cfg_s, want_importance=False)
        except Exception as _e:
            print(f'  lambda={_lam_s}: FAILED {repr(_e)[:70]}'); continue
        _oos_s = [r for r in _rows_s if r['sample'] == 'oos']
        for r in _oos_s:                       # tag + keep in the SEPARATE appendix file
            r2 = dict(r); r2['lambda_grid_value'] = float(_lam_s)
            _ls_rows.append(r2)
        if _oos_s:
            _mDA  = float(np.mean([r['dir_acc'] for r in _oos_s]))
            _mCT  = float(np.mean([r['ct_r2_oos'] for r in _oos_s]))
            _ls_summary.append(dict(lambda_=float(_lam_s), mean_oos_dir_acc=_mDA,
                                    mean_ct_r2=_mCT, n_folds=len(_oos_s)))
            _frozen_mark = '  (= FROZEN_CONFIG lam)' if abs(_lam_s - LAM) < 1e-12 else ''
            print(f'  lambda={_lam_s:<6} mean OOS DirAcc={_mDA:.4f}  '
                  f'mean CT-R²={_mCT:+.4f}  n_folds={len(_oos_s)}{_frozen_mark}')

    lambda_sens_summary = pd.DataFrame(_ls_summary)
    if len(_ls_rows):
        # SEPARATE appendix file — NOT v5_metrics_long.csv, NOT in any MTC family.
        pd.DataFrame(_ls_rows).to_csv(f'{OUT_DIR}/v5_lambda_sensitivity.csv', index=False)
        print(f'\n  Saved appendix: {OUT_DIR}/v5_lambda_sensitivity.csv '
              f'({len(_ls_rows)} OOS rows; excluded from primary metrics + MTC)')
    if len(lambda_sens_summary):
        _robust = bool((lambda_sens_summary['mean_oos_dir_acc'].sub(0.5).abs() < 0.03).all()
                       and (lambda_sens_summary['mean_ct_r2'] < 0.01).all())
        print(f'\n  Verdict: null is {"ROBUST" if _robust else "NOT uniformly robust"} '
              f'across lambda — OOS DirAcc ≈ 0.50 and CT-R² ≈ 0 at every λ '
              f'(λ stays fixed a priori at {LAM}; this is a robustness check, not a selection).')

### Part C — Hiking-period OOS export
Export out-of-sample predictions for the hiking regime alongside a par carry/roll decomposition for downstream trading analysis. Export only: carry/roll is not a model feature and these predictions never feed model selection.

In [ ]:
# ── PART C (v5): Hiking-first OOS trading export — parametrized by H ──────────
# EXPORT-ONLY. Carry/roll is NOT a model feature. Predictions are out-of-sample
# (the model never trains on the Hiking window). Uses the FROZEN config + the v5
# PCA fit path. Runs per horizon in PARTC_HORIZONS so the export matches the swept
# horizon. Block-CV/LORO predictions NEVER enter this cell (causal walk-forward only).
HIKE_START, HIKE_END = pd.Timestamp('2022-01-01'), pd.Timestamp('2026-12-31')
PARTC_HORIZONS = HORIZONS if not QUICK_MODE else QUICK_HGRID


def _spot_clamped(country, T, df):
    cols = {MATURITY_YEARS[m]: df[f'{country}_{m}']
            for m in MATURITY_YEARS if f'{country}_{m}' in df.columns}
    if not cols:
        return pd.Series(np.nan, index=df.index)
    pts = sorted(cols)
    if T <= pts[0]:
        return cols[pts[0]]
    if T >= pts[-1]:
        return cols[pts[-1]]
    for i in range(len(pts) - 1):
        if pts[i] <= T <= pts[i + 1]:
            w = (T - pts[i]) / (pts[i + 1] - pts[i])
            return cols[pts[i]] * (1 - w) + cols[pts[i + 1]] * w
    return pd.Series(np.nan, index=df.index)


def carry_roll_decomp_h(sec, df, country, maturity, h):
    """Par carry+roll on the SPOT curve, scaled to horizon h. EXPORT-ONLY."""
    M = MATURITY_YEARS.get(maturity, np.nan)
    Hy = h / 252.0
    if np.isnan(M):
        return None
    s_M   = df[sec]
    s_MmH = _spot_clamped(country, M - Hy, df)
    roll  = s_M - s_MmH
    short = (df['nor_nowa_level'].copy() if 'nor_nowa_level' in df.columns
             else pd.Series(np.nan, index=df.index))
    ibor_col = COUNTRY_PRIMARY_IBOR.get(country, (None, None))[0]
    if ibor_col and ibor_col in df.columns:
        short = short.fillna(df[ibor_col])
    carry = (s_M - short) * Hy
    return pd.DataFrame({'date': df.index, 's_spot_M': s_M.values,
                         's_spot_M_minus_H': s_MmH.values, 'short_funding': short.values,
                         'carry': carry.values, 'roll_down': roll.values,
                         'carry_roll_total': (carry + roll).values})


def run_security_hiking_export_v5(sec, df_raw, h, cfg):
    if sec not in df_raw.columns:
        return None
    country, maturity = sec.rsplit('_', 1)
    panel = build_panel_v4(df_raw, [sec], h)
    if len(panel) == 0:
        return None
    fc = [c for c in panel.columns if c not in META_COLS]
    purge_ts = HIKE_START - pd.tseries.offsets.BDay(h - 1)
    tr = panel[panel['date'] < purge_ts].copy()
    te = panel[(panel['date'] >= HIKE_START) & (panel['date'] <= HIKE_END)].copy()
    if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
        return None
    x_tr, x_te, _ = fit_transform_xm_pca(tr[fc], te[fc])
    _, _, yhat, _, _, _ = fit_htboost_v5(tr['y'].to_numpy(float), x_tr, tr['date'], x_te, cfg, h)
    out = pd.DataFrame({
        'date': te['date'].values, 'security': sec, 'tenor': maturity, 'regime': 'Hiking',
        'level_t': te['level'].values, 'carry_roll': te['carry_roll'].values,
        'y_true': te['y'].values, 'y_pred': yhat})
    out['signal']  = np.sign(out['y_pred'])
    out['hit']     = (np.sign(out['y_pred']) == np.sign(out['y_true'])).astype(int)
    out['horizon'] = h
    out['n_train'] = len(tr)
    dec = carry_roll_decomp_h(sec, df_raw, country, maturity, h)
    if dec is not None:
        out = out.merge(dec, on='date', how='left')
    return out


if not RUN_WALK_FORWARD:
    print('[Part C skipped] RUN_WALK_FORWARD = False')
else:
    for h in PARTC_HORIZONS:
        _purge = HIKE_START - pd.tseries.offsets.BDay(h - 1)
        print(f'PART C — Hiking OOS export, H={h}  (train < {_purge.date()})')
        parts = []
        for sec in SWEEP_TENORS:
            r = run_security_hiking_export_v5(sec, df_raw, h, FROZEN_CONFIG)
            if r is None:
                print(f'  {sec:<10s} skipped'); continue
            parts.append(r)
            print(f'  {sec:<10s} OOS n={len(r):4d}  DirAcc={r["hit"].mean():.3f}')
        if not parts:
            print('  (no tenor produced a Hiking fold at this H)'); continue
        hike_export = pd.concat(parts, ignore_index=True)
        _cols = ['date', 'security', 'tenor', 'regime', 'level_t', 'carry_roll', 'y_true',
                 'y_pred', 'signal', 'hit', 'horizon', 'n_train', 's_spot_M',
                 's_spot_M_minus_H', 'short_funding', 'carry', 'roll_down', 'carry_roll_total']
        _cols = [c for c in _cols if c in hike_export.columns]
        hike_export[_cols].sort_values(['date', 'security']).to_csv(
            f'{OUT_DIR}/v5_nor_hiking_predictions_H{h}.csv', index=False)

        def _pivot_save(val, fname):
            p = hike_export.pivot_table(index='date', columns='tenor', values=val, aggfunc='first')
            p = p.reindex(columns=[t for t in ['1Y', '5Y', '10Y', '15Y', '30Y'] if t in p.columns])
            p.to_csv(f'{OUT_DIR}/{fname}')

        _pivot_save('y_pred',           f'v5_nor_hiking_signal_matrix_H{h}.csv')
        _pivot_save('y_true',           f'v5_nor_hiking_realized_matrix_H{h}.csv')
        _pivot_save('carry_roll_total', f'v5_nor_hiking_carry_matrix_H{h}.csv')
        print(f'  saved v5_nor_hiking_*_H{h}.csv ({len(hike_export)} rows)')
    print('\n[PART C COMPLETE] OOS Hiking predictions + par carry/roll exported per H. '
          'DV01 intentionally excluded (duration guard).')

### Block-CV / leave-one-regime-out runner
A deliberately non-causal robustness diagnostic that trains on data surrounding each held-out block (including the future) with a two-sided purge and embargo. It measures regime transferability, not forecast skill, and never enters the trading export.

In [ ]:
# ── block-CV / leave-one-regime-out runner (NON-CAUSAL diagnostic) ──────
# Trains on data SURROUNDING the held-out block — INCLUDING periods in the FUTURE
# relative to the test block. This deliberately breaks causal ordering: it is a
# learnability / regime-transfer diagnostic, NOT a forecast (see the title cell's
# ex-ante asymmetric interpretation rule). Reuses the FROZEN config
# verbatim — NO hyperparameter selection happens here.
#
# Two-sided purge+embargo (the leakage trap): each test block is surrounded by
# training data on BOTH sides, so we drop training rows within
# gap = OVERLAP + EMBARGO = (H-1) + max(H,10) business days of EVERY segment
# boundary (both edges; every seam for LORO). Symmetric over-purging removes a
# little valid training data on the no-overlap side — intentionally conservative
# for a null harness. Note: point-in-time features at the first rows of a test
# block legitimately read the preceding (training) block's calendar history —
# that is backward-looking and CORRECT, not a leak.

def _blockcv_entries(scheme):
    """Return [(label, regime, [(start_ts, end_ts), ...]), ...] for the scheme."""
    if scheme == 'block_cv':
        return [(name, regime, [(pd.Timestamp(s), pd.Timestamp(e))])
                for (name, s, e, regime) in BLOCK_CV_FOLDS]
    if scheme == 'loro':
        out = []
        for regime in dict.fromkeys(r for (_n, _s, _e, r) in BLOCK_CV_FOLDS):
            segs = [(pd.Timestamp(s), pd.Timestamp(e))
                    for (_n, s, e, r) in BLOCK_CV_FOLDS if r == regime]
            out.append((f'LORO_{regime}', regime, segs))
        return out
    raise ValueError(scheme)


def run_security_blockcv(sec, df_raw, scheme, H, cfg, want_importance=False, verbose=False):
    """Block-CV / LORO across blocks for one security at horizon H. Two-sided
    purge+embargo around every test segment. Emits train+oos rows per block."""
    if sec not in df_raw.columns:
        return [], [], [], []
    panel = build_panel_v4(df_raw, [sec], H)
    if len(panel) == 0:
        return [], [], [], []
    fc  = [c for c in panel.columns if c not in META_COLS]
    gap = (H - 1) + EMBARGO_FOR_H(H)                       # business days, scales with H
    dates = panel['date']
    rows, imps, taus, preds_all = [], [], [], []
    for label, regime, segs in _blockcv_entries(scheme):
        test_mask = pd.Series(False, index=panel.index)
        for (s, e) in segs:
            test_mask |= (dates >= s) & (dates <= e)
        train_mask = ~test_mask
        for (s, e) in segs:                                # symmetric gap, both edges
            lo = s - pd.tseries.offsets.BDay(gap)
            hi = e + pd.tseries.offsets.BDay(gap)
            train_mask &= ~((dates >= lo) & (dates <= hi))
        tr = panel[train_mask].copy()
        te = panel[test_mask].copy()
        if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
            if verbose:
                print(f'  {scheme}/{label}: skip (train={len(tr)}, test={len(te)})')
            continue
        try:
            r, imp, tau, preds = _fit_score_rows(tr, te, fc, H, cfg, sec, scheme, label, regime,
                                        want_importance=want_importance, want_tau=True)
        except Exception as ex:
            if verbose:
                print(f'  {scheme}/{label}: FAILED {repr(ex)[:70]}')
            continue
        rows.extend(r)
        preds_all.append(preds)
        if imp is not None:
            imps.append({'security': sec, 'horizon': H, 'fold': label, 'imp': imp})
        if tau is not None:
            _country, _tenor = sec.rsplit('_', 1)
            taus.append({'security': sec, 'country': _country, 'tenor': _tenor,
                         'horizon': H, 'fold': label, 'regime': regime,
                         'tau_w': tau['tau_w'], 'tau_w_band': tau['tau_w_band'], 'table': tau['table']})
        if verbose:
            oos = r[1]
            print(f'  {scheme}/{label:<12s} train DA={r[0]["dir_acc"]:.3f}  '
                  f'OOS DA={oos["dir_acc"]:.3f}  n_train={len(tr)}  n_test={oos["n_obs"]}')
    return rows, imps, taus, preds_all


print('Block-CV/LORO runner defined: run_security_blockcv (two-sided purge+embargo,')
print('  reuses FROZEN_CONFIG; non-causal learnability diagnostic — NOT a forecast)')

### Block-CV / LORO sweep
Run the non-causal diagnostic across schemes, horizons, tenors and blocks using the same frozen config. Results feed a separate multiple-testing family and the robustness section only.

In [ ]:
# ── BLOCK-CV / LORO SWEEP — schemes × H × tenors × blocks ───────────────
# Heavy cell (same fit cost as walk-forward). Gated by RUN_BLOCK_CV / RUN_LORO.
# Reuses FROZEN_CONFIG verbatim. Results feed a SEPARATE MTC family and the
# robustness section — never Part C, never the headline.
block_rows, block_imps, block_taus, block_preds = [], [], [], []
_block_failed = []
_schemes = ([] + (['block_cv'] if RUN_BLOCK_CV else []) + (['loro'] if RUN_LORO else []))

if not _schemes:
    print('[skipped] RUN_BLOCK_CV = RUN_LORO = False')
else:
    _t0 = time.time()
    for scheme in _schemes:
        for H_run in HORIZONS:
            for sec in SWEEP_TENORS:
                _ts = time.time()
                try:
                    rows, imps, taus, preds = run_security_blockcv(sec, df_raw, scheme, H_run,
                                                      FROZEN_CONFIG, want_importance=True)
                except Exception as e:
                    print(f'  {scheme} H={H_run} {sec}: FAILED {repr(e)[:60]}')
                    _block_failed.append((scheme, H_run, sec)); continue
                block_rows.extend(rows); block_imps.extend(imps); block_taus.extend(taus); block_preds.extend(preds)
                _oos = [r for r in rows if r['sample'] == 'oos']
                _da = ', '.join(f'{r["dir_acc"]:.3f}' for r in _oos)
                print(f'  {scheme:<9s} H={H_run:<3d} {sec:<10s} blocks={len(_oos)} '
                      f'OOS_DA=[{_da}]  ({time.time()-_ts:.1f}s)')
                pd.DataFrame(block_rows).to_csv(f'{OUT_DIR}/_block_rows_partial.csv', index=False)
    print(f'\nBlock-CV/LORO sweep done: {len(block_rows)} rows in {(time.time()-_t0)/60:.1f} min')
    if _block_failed:
        print(f'  {len(_block_failed)} cells failed: {_block_failed}')

# Block-CV/LORO sidecars (monolithic - match block-CV's metric persistence; NOT machine-stamped).
if block_preds:
    pd.concat(block_preds, ignore_index=True).to_csv(f'{OUT_DIR}/v5_blockcv_preds.csv', index=False)
if block_taus:
    import pickle as _pkl_bc
    with open(f'{OUT_DIR}/v5_blockcv_tau.pkl', 'wb') as _fh_bc:
        _pkl_bc.dump(block_taus, _fh_bc)
    pd.DataFrame([{k: v for k, v in _t.items() if k != 'table'} for _t in block_taus]
                 ).to_csv(f'{OUT_DIR}/v5_blockcv_tau.csv', index=False)
df_block = pd.DataFrame(block_rows)
print(f'df_block shape: {df_block.shape}')

### Assemble results and apply multiple-testing correction
Concatenate all rows into the tidy long CSV and apply Bonferroni and BH-FDR within two separate families — walk-forward (forecastability) and non-causal (learnability) — since they test different hypotheses.

In [ ]:
# ── assemble the tidy long CSV + apply MTC in SEPARATE families ──
# Walk-forward and block-CV/LORO are corrected in their OWN families (they test
# forecastability vs learnability). The shared schema (SHARED_COLS) is preserved so
# the pooled notebook merges with zero reconciliation.
_parts = [d for d in (df_wf, df_block) if len(d) > 0]
df_all = pd.concat(_parts, ignore_index=True) if _parts else pd.DataFrame(columns=SHARED_COLS)

# Ensure every shared column exists (pooled-comparability contract).
for c in SHARED_COLS:
    if c not in df_all.columns:
        df_all[c] = np.nan
for c in ('reject_bonferroni', 'reject_fdr_bh', 'mtc_N', 'mtc_family'):
    if c not in df_all.columns:
        df_all[c] = (False if 'reject' in c else (np.nan if c == 'mtc_N' else ''))


def apply_mtc_family(df, schemes, label):
    """Bonferroni + BH-FDR over the OOS binomial p-values within one MTC family.
    Returns (N, n_bonf, n_bh) and writes reject_* / mtc_N / mtc_family in place."""
    mask = (df['sample'] == 'oos') & df['validation_scheme'].isin(schemes) \
           & df['binom_pval'].notna()
    idx = df.index[mask]
    if len(idx) == 0:
        return 0, 0, 0
    pvals = df.loc[idx, 'binom_pval'].to_numpy(float)
    rej_b, _, _, _ = multipletests(pvals, alpha=ALPHA_MT, method='bonferroni')
    rej_h, _, _, _ = multipletests(pvals, alpha=ALPHA_MT, method='fdr_bh')
    df.loc[idx, 'reject_bonferroni'] = rej_b
    df.loc[idx, 'reject_fdr_bh']     = rej_h
    df.loc[idx, 'mtc_N']             = len(idx)
    df.loc[idx, 'mtc_family']        = label
    return len(idx), int(rej_b.sum()), int(rej_h.sum())


# Family 1 — walk-forward {horizon × tenor × regime}
N_wf, b_wf, h_wf = apply_mtc_family(
    df_all, ['walk_forward'], 'walk_forward:{horizon×tenor×regime}')
# Family 2 — non-causal {scheme × horizon × tenor × block}  (block_cv ∪ loro)
N_bl, b_bl, h_bl = apply_mtc_family(
    df_all, ['block_cv', 'loro'], 'noncausal:{scheme×horizon×tenor×block}')

# Order columns: shared first, then extras.
_extra = [c for c in df_all.columns if c not in SHARED_COLS]
df_all = df_all[[c for c in SHARED_COLS if c in df_all.columns] + _extra]
df_all.to_csv(f'{OUT_DIR}/v5_metrics_long.csv', index=False)

print('=== v5_metrics_long.csv written ===')
print(f'  rows={len(df_all)}  cols={df_all.shape[1]}  (shared={len(SHARED_COLS)})')
print(f'  walk-forward MTC family : N={N_wf}  Bonferroni survivors={b_wf}  BH-FDR={h_wf}')
print(f'  non-causal  MTC family : N={N_bl}  Bonferroni survivors={b_bl}  BH-FDR={h_bl}')
print(f'  Saved: {OUT_DIR}/v5_metrics_long.csv')

### Report — overfit-gap table and plots
Pair train and OOS metrics with an explicit overfit-gap column, aggregate by horizon and regime, and plot the train-vs-OOS distributions. This is the central in-sample-vs-OOS comparison.

In [ ]:
# ── report — master overfit-gap table, aggregates, plots ─────────────────
# The Ch.7-vs-Ch.8 headline: train and OOS DirAcc/R² side by side with an explicit
# overfit-gap column. Walk-forward only here (the causal finding).
def _pair_train_oos(df, schemes):
    d = df[df['validation_scheme'].isin(schemes)].copy()
    keys = ['security', 'tenor', 'horizon', 'fold', 'regime']
    piv = d.pivot_table(index=keys, columns='sample',
                        values=['dir_acc', 'r2_raw', 'ct_r2_oos', 'n_obs'], aggfunc='first')
    piv.columns = [f'{a}_{b}' for a, b in piv.columns]
    piv = piv.reset_index()
    if 'dir_acc_train' in piv and 'dir_acc_oos' in piv:
        piv['gap_dir_acc'] = piv['dir_acc_train'] - piv['dir_acc_oos']
    if 'r2_raw_train' in piv and 'r2_raw_oos' in piv:
        piv['gap_r2'] = piv['r2_raw_train'] - piv['r2_raw_oos']
    return piv


master_wf = _pair_train_oos(df_all, ['walk_forward'])
master_wf.to_csv(f'{OUT_DIR}/v5_master_overfit_gap.csv', index=False)
print('=== master overfit-gap table (walk-forward) — head ===')
_show = [c for c in ['security', 'horizon', 'fold', 'regime', 'dir_acc_train',
                     'dir_acc_oos', 'gap_dir_acc', 'ct_r2_oos_oos', 'n_obs_oos']
         if c in master_wf.columns]
if len(master_wf):
    print(master_wf[_show].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f'\n  Saved: {OUT_DIR}/v5_master_overfit_gap.csv')

# Aggregates by horizon and by regime (OOS).
oos_wf = df_all[(df_all['validation_scheme'] == 'walk_forward') & (df_all['sample'] == 'oos')]
if len(oos_wf):
    print('\n=== Aggregate by HORIZON (walk-forward OOS) ===')
    agg_h = oos_wf.groupby('horizon').agg(
        n=('dir_acc', 'size'), mean_DA=('dir_acc', 'mean'), std_DA=('dir_acc', 'std'),
        pct_gt50=('dir_acc', lambda s: float((s > 0.5).mean() * 100)),
        mean_CT_R2=('ct_r2_oos', 'mean'),
        bonf=('reject_bonferroni', 'sum'), bh=('reject_fdr_bh', 'sum'))
    print(agg_h.to_string(float_format=lambda x: f'{x:.3f}'))
    agg_h.to_csv(f'{OUT_DIR}/v5_agg_by_horizon.csv')

    print('\n=== Aggregate by REGIME (walk-forward OOS) ===')
    agg_r = oos_wf.groupby('regime').agg(
        n=('dir_acc', 'size'), mean_DA=('dir_acc', 'mean'), std_DA=('dir_acc', 'std'),
        pct_gt50=('dir_acc', lambda s: float((s > 0.5).mean() * 100)),
        mean_CT_R2=('ct_r2_oos', 'mean'))
    print(agg_r.to_string(float_format=lambda x: f'{x:.3f}'))
    agg_r.to_csv(f'{OUT_DIR}/v5_agg_by_regime.csv')

# ── Plots ────────────────────────────────────────────────────────────────────
if len(master_wf) and {'dir_acc_train', 'dir_acc_oos'} <= set(master_wf.columns):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    ax = axes[0, 0]
    ax.hist(master_wf['dir_acc_train'].dropna(), bins=15, alpha=0.6, label='train', color='steelblue')
    ax.hist(master_wf['dir_acc_oos'].dropna(),   bins=15, alpha=0.6, label='OOS',   color='darkorange')
    ax.axvline(0.5, color='red', ls='--'); ax.set_title('Train vs OOS DirAcc'); ax.legend()
    ax.set_xlabel('DirAcc')

    ax = axes[0, 1]
    g = master_wf.groupby('horizon')['gap_dir_acc'].mean()
    ax.plot(g.index, g.values, 'o-', color='navy')
    ax.axhline(0, color='grey', ls=':'); ax.set_title('Overfit gap (train−OOS DirAcc) vs horizon')
    ax.set_xlabel('horizon (days)'); ax.set_ylabel('gap')

    ax = axes[1, 0]
    if len(oos_wf):
        m = oos_wf.groupby('horizon')['dir_acc'].mean()
        s = oos_wf.groupby('horizon')['dir_acc'].std()
        ax.errorbar(m.index, m.values, yerr=s.values, fmt='o-', capsize=4, color='darkgreen')
    ax.axhline(0.5, color='red', ls='--'); ax.set_title('OOS DirAcc vs horizon (±1 SD)')
    ax.set_xlabel('horizon (days)'); ax.set_ylabel('OOS DirAcc')

    ax = axes[1, 1]
    if len(oos_wf):
        ax.scatter(oos_wf['cw_pval'], oos_wf['dmh_pval'], alpha=0.5, color='purple')
        ax.axvline(0.05, color='red', ls=':'); ax.axhline(0.05, color='red', ls=':')
    ax.set_title('CW vs DM-Harvey one-sided p-values (OOS)')
    ax.set_xlabel('Clark–West p'); ax.set_ylabel('DM-Harvey p')
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/v5_report_panels.png', dpi=120); plt.show()
    print(f'\n  Saved figure: {OUT_DIR}/v5_report_panels.png')

### Report — feature importance and block-CV comparison
Aggregate bucketed feature importance by horizon, place block-CV OOS next to walk-forward OOS under the asymmetric interpretation rule, and write the self-contained Markdown report.

In [ ]:
# ── report — feature-importance buckets, block-CV vs walk-forward, md ─
def _agg_importance(imps):
    """Average bucketed feature importance across securities/folds, per horizon."""
    recs = []
    for r in imps:
        b = {}
        for feat, val in r['imp'].items():
            bk = bucket_feature(feat)
            b[bk] = b.get(bk, 0.0) + float(val)
        b['horizon'] = r['horizon']
        recs.append(b)
    if not recs:
        return pd.DataFrame()
    d = pd.DataFrame(recs).fillna(0.0)
    return d.groupby('horizon').mean(numeric_only=True)


imp_buckets = _agg_importance(wf_imps)
if len(imp_buckets):
    print('=== Feature importance by bucket (walk-forward, avg across securities, per horizon) ===')
    print(imp_buckets.to_string(float_format=lambda x: f'{x:.1f}'))
    imp_buckets.to_csv(f'{OUT_DIR}/v5_importance_buckets.csv')
    # Macro-vs-curve contrast (Bianchi et al.)
    for col in ('macro', 'curve', 'cross_market', 'norway'):
        if col in imp_buckets.columns:
            print(f'  mean {col:<12s} importance: {imp_buckets[col].mean():.1f}')
else:
    print('No feature-importance records (sweep skipped or HTBrelevance unavailable).')

# ── Block-CV OOS next to walk-forward OOS, with the gap column ────────────────
def _scheme_oos_mean(df, scheme):
    d = df[(df['sample'] == 'oos') & (df['validation_scheme'] == scheme)]
    if len(d) == 0:
        return pd.Series(dtype=float)
    return d.groupby(['tenor', 'regime'])['dir_acc'].mean()


cmp_block = None
wf_m = _scheme_oos_mean(df_all, 'walk_forward')
bl_m = _scheme_oos_mean(df_all, 'block_cv')
lo_m = _scheme_oos_mean(df_all, 'loro')
if len(wf_m):
    cmp_block = pd.DataFrame({'walk_forward_OOS': wf_m})
    if len(bl_m):
        cmp_block['block_cv_OOS'] = bl_m
        cmp_block['gap_block_minus_wf'] = cmp_block['block_cv_OOS'] - cmp_block['walk_forward_OOS']
    if len(lo_m):
        cmp_block['loro_OOS'] = lo_m
    cmp_block = cmp_block.reset_index()
    cmp_block.to_csv(f'{OUT_DIR}/v5_blockcv_vs_walkforward.csv', index=False)
    print('\n=== Block-CV / LORO OOS vs walk-forward OOS (mean DirAcc by tenor×regime) ===')
    print(cmp_block.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    print('\n  Interpretation rule (ex ante, asymmetric):')
    print('   • block-CV ≈ 0.50            → STRONGER null confirmation (future data transfers nothing)')
    print('   • block-CV > wf while wf≈0.50 → future-information / training-size artefact, NON-stationarity,')
    print('                                  NOT tradeable skill (never enters Part C).')

# ── self-contained v5_report.md ──────────────────────────────────────────────
def _md_table(df, floatfmt='{:.3f}', maxrows=40):
    if df is None or len(df) == 0:
        return '_(no rows)_\n'
    d = df.copy()
    if isinstance(d.index, pd.MultiIndex) or d.index.name:
        d = d.reset_index()
    d = d.head(maxrows)
    cols = list(d.columns)
    def fmt(v):
        return floatfmt.format(v) if isinstance(v, (int, float, np.floating)) and not isinstance(v, bool) and pd.notna(v) else str(v)
    out = ['| ' + ' | '.join(map(str, cols)) + ' |',
           '| ' + ' | '.join('---' for _ in cols) + ' |']
    for _, r in d.iterrows():
        out.append('| ' + ' | '.join(fmt(r[c]) for c in cols) + ' |')
    return '\n'.join(out) + '\n'


_lines = []
_lines.append('# HTBoost v5 — Results report\n')
_lines.append(f'_Generated {RUN_TS} · notebook={NOTEBOOK_TAG} · config_hash={FROZEN_HASH}_\n')
_lines.append(f'Frozen config (inner-CV selected on train only): `{FROZEN_CONFIG}`  ·  '
              f'PCA xm_* → ≤{XM_PCA_KMAX} comps (≥{XM_PCA_VAR:.0%} train var)\n')
_lines.append('\n## Decision rule (ex ante)\n')
_lines.append('Null confirmed if OOS DirAcc is noise-consistent and no security survives MTC. '
              'A genuine lead requires MTC survival AND multi-regime positivity (not Hiking-only).\n')
_lines.append('\n## Walk-forward — aggregate by horizon (OOS)\n')
_lines.append(_md_table(agg_h if 'agg_h' in dir() else None))
_lines.append('\n## Walk-forward — aggregate by regime (OOS)\n')
_lines.append(_md_table(agg_r if 'agg_r' in dir() else None))
_lines.append('\n## Multiple-testing survivors (separate families)\n')
_lines.append(f'- Walk-forward `{{horizon×tenor×regime}}`: N={N_wf}, Bonferroni={b_wf}, BH-FDR={h_wf}\n')
_lines.append(f'- Non-causal `{{scheme×horizon×tenor×block}}`: N={N_bl}, Bonferroni={b_bl}, BH-FDR={h_bl}\n')
_lines.append('\n## Feature importance by bucket (per horizon)\n')
_lines.append(_md_table(imp_buckets, floatfmt='{:.1f}'))
_lines.append('\n## Block-CV / LORO vs walk-forward (asymmetric interpretation)\n')
_lines.append('block-CV is a NON-causal learnability diagnostic, never a forecast and never in Part C. '
              'block-CV≈0.50 ⇒ stronger null; block-CV>wf ⇒ future-info/size artefact (non-stationarity), not skill.\n')
_lines.append(_md_table(cmp_block))
_lines.append('\n## Figures\n')
_lines.append(f'![panels](v5_report_panels.png)\n')
_lines.append('\n## Appendix — master overfit-gap table (head)\n')
_lines.append(_md_table(master_wf[_show] if len(master_wf) else None))
with open(f'{OUT_DIR}/v5_report.md', 'w') as _f:
    _f.write('\n'.join(_lines))
print(f'\n[REPORT] wrote {OUT_DIR}/v5_report.md  (+ supporting CSVs/PNGs)')

### τ^w smoothness diagnostic — persistence & figures
Importance-weighted smoothness $\tau^{w}$ (HTBoost eq. 19) extracted via `HTBweightedtau`, persisted as a model diagnostic (separate from the metrics schema), plus the three thesis figures (heatmap, taxonomy importance, PDP panel).

In [ ]:
# ── τ^w persistence — per-fold scalar + per-feature avgtau (DIAGNOSTIC, not metrics)
# τ^w is a model smoothness diagnostic, deliberately kept OUT of the shared
# htb_metrics long schema (v5_metrics_long.csv) so the pooled-comparability contract
# is untouched. Two separate sidecar files under OUT_DIR, both long-format with
# identifier columns (matching the notebook's metrics convention):
#   • v5_tau_w_by_fold.csv          one row per security×horizon×fold  (scalar τ^w + band)
#   • v5_tau_avgtau_perfeature.csv  one row per security×horizon×fold×feature (avgtau)
# NOTE: τ^w is only interpretable under modality=:accurate/:compromise and in large
# samples; early/narrow-coverage walk-forward folds (e.g. post-GFC) are NOISY.
TAU_W_BY_FOLD_CSV   = f'{OUT_DIR}/v5_tau_w_by_fold.csv'
TAU_PERFEAT_CSV     = f'{OUT_DIR}/v5_tau_avgtau_perfeature.csv'

_tau_scalar_rows, _tau_feat_rows = [], []
for _rec in (wf_taus if 'wf_taus' in dir() else []):
    _tau_scalar_rows.append({k: _rec[k] for k in
                             ('security', 'country', 'tenor', 'horizon', 'fold',
                              'regime', 'tau_w', 'tau_w_band')})
    _tbl = _rec.get('table')
    if _tbl is not None and len(_tbl):
        _t = _tbl.copy()
        _t.insert(0, 'security', _rec['security']); _t.insert(1, 'tenor', _rec['tenor'])
        _t.insert(2, 'horizon', _rec['horizon']);   _t.insert(3, 'fold', _rec['fold'])
        _tau_feat_rows.append(_t)

df_wf_tau = pd.DataFrame(_tau_scalar_rows)
if len(df_wf_tau):
    df_wf_tau = df_wf_tau.sort_values(['horizon', 'tenor', 'fold']).reset_index(drop=True)
    df_wf_tau.to_csv(TAU_W_BY_FOLD_CSV, index=False)
    df_tau_perfeat = pd.concat(_tau_feat_rows, ignore_index=True) if _tau_feat_rows else pd.DataFrame()
    if len(df_tau_perfeat):
        df_tau_perfeat.to_csv(TAU_PERFEAT_CSV, index=False)
    print('=== τ^w by fold (walk-forward, diagnostic) ===')
    print(df_wf_tau.to_string(index=False, float_format=lambda x: f'{x:.2f}'))
    print(f'\n  Saved {TAU_W_BY_FOLD_CSV}  ({len(df_wf_tau)} rows)')
    print(f'  Saved {TAU_PERFEAT_CSV}  ({len(df_tau_perfeat)} per-feature rows)')
    _band_counts = df_wf_tau['tau_w_band'].value_counts()
    print(f'  band counts: {_band_counts.to_dict()}')
else:
    df_tau_perfeat = pd.DataFrame()
    print('[τ^w] no per-fold records yet — run the walk-forward sweep (cell with '
          'want_tau=True) first, or use the diagnostic fit cell below for a single fold.')


In [ ]:
# ── τ^w / importance / PDP plotting helpers (matplotlib only — no Julia plotting)
# All three operate on plain numeric arrays returned by the HTBoost bridge
# (tau_w, importance, q, pdp), so they are unit-testable without the Julia runtime.
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

TENOR_ORDER   = ['1Y', '5Y', '10Y', '15Y', '30Y']
TAU_BAND_COLS = {'strong': OKABE_ITO[2], 'mild': OKABE_ITO[0], 'weak': OKABE_ITO[1],
                 'none': OKABE_ITO[3], 'na': '#cccccc'}
_BAND_SEQ     = ['strong', 'mild', 'weak', 'none', 'na']
_BAND_CODE    = {b: i for i, b in enumerate(_BAND_SEQ)}


def plot_tau_heatmap(df_tau, fold_name, out_path, tenor_order=TENOR_ORDER):
    """τ^w summary heatmap: rows=tenors, cols=horizons, cell shaded by band and
    annotated with the numeric τ^w. `df_tau` is already filtered to ONE fold."""
    tenors = [t for t in tenor_order if t in set(df_tau['tenor'])]
    horizons = sorted(df_tau['horizon'].unique())
    val = df_tau.pivot_table(index='tenor', columns='horizon', values='tau_w', aggfunc='mean')
    bnd = df_tau.pivot_table(index='tenor', columns='horizon', values='tau_w_band',
                             aggfunc='first')
    code = np.full((len(tenors), len(horizons)), _BAND_CODE['na'], dtype=int)
    for i, t in enumerate(tenors):
        for j, h in enumerate(horizons):
            b = bnd.loc[t, h] if (t in bnd.index and h in bnd.columns) else 'na'
            code[i, j] = _BAND_CODE.get(b if isinstance(b, str) else 'na', _BAND_CODE['na'])
    cmap = ListedColormap([TAU_BAND_COLS[b] for b in _BAND_SEQ])
    norm = BoundaryNorm(np.arange(-0.5, len(_BAND_SEQ) + 0.5), cmap.N)
    fig, ax = plt.subplots(figsize=(1.6 + 1.1 * len(horizons), 1.2 + 0.7 * len(tenors)))
    ax.imshow(code, cmap=cmap, norm=norm, aspect='auto')
    ax.set_xticks(range(len(horizons))); ax.set_xticklabels([f'{h}d' for h in horizons])
    ax.set_yticks(range(len(tenors)));   ax.set_yticklabels(tenors)
    ax.set_xlabel('forecast horizon'); ax.set_ylabel('NOK swap tenor')
    for i, t in enumerate(tenors):
        for j, h in enumerate(horizons):
            v = val.loc[t, h] if (t in val.index and h in val.columns) else np.nan
            txt = f'{v:.1f}' if np.isfinite(v) else '—'
            ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                    color='white' if code[i, j] in (0, 3) else 'black')
    ax.set_title('Importance-weighted smoothness $\\tau^{w}$ by tenor × horizon')
    handles = [Patch(facecolor=TAU_BAND_COLS[b], label=lbl) for b, lbl in
               [('strong', 'strong ≤7'), ('mild', 'mild 7–15'),
                ('weak', 'weak 15–25'), ('none', 'none >25')]]
    ax.legend(handles=handles, bbox_to_anchor=(1.02, 1.0), loc='upper left',
              frameon=False, fontsize=8, title='band')
    fig.savefig(out_path, format='pdf'); plt.show(); plt.close(fig)
    return out_path


def plot_importance_by_taxonomy(imp_records, out_path, per_tenor=False):
    """Aggregate HTBrelevance importance into the 8 taxonomy buckets (bucket_feature).
    `imp_records` is a list of {'security','horizon','fold','imp':{feat:val}}.
    per_tenor=False → single pooled bar chart; True → grouped bars per tenor."""
    rows = []
    for r in imp_records:
        tenor = r['security'].rsplit('_', 1)[1]
        for feat, val in r['imp'].items():
            rows.append({'tenor': tenor, 'bucket': bucket_feature(feat), 'imp': float(val)})
    d = pd.DataFrame(rows)
    if not len(d):
        return None
    fig, ax = plt.subplots(figsize=(9, 4.5))
    if per_tenor:
        piv = d.pivot_table(index='bucket', columns='tenor', values='imp', aggfunc='mean').fillna(0.0)
        piv = piv.reindex(columns=[t for t in TENOR_ORDER if t in piv.columns])
        piv = piv.loc[piv.sum(axis=1).sort_values(ascending=False).index]
        piv.plot(kind='bar', ax=ax, width=0.8)
        ax.legend(title='tenor', frameon=False, fontsize=8)
    else:
        agg = d.groupby('bucket')['imp'].mean().sort_values(ascending=False)
        ax.bar(range(len(agg)), agg.values, color=OKABE_ITO[0])
        ax.set_xticks(range(len(agg))); ax.set_xticklabels(agg.index, rotation=35, ha='right')
    ax.set_ylabel('mean HTBrelevance importance'); ax.set_xlabel('feature taxonomy bucket')
    ax.set_title('HTBoost feature importance by taxonomy bucket (curve vs macro)')
    fig.tight_layout(); fig.savefig(out_path, format='pdf'); plt.show(); plt.close(fig)
    return out_path


def plot_pdp_panel(q, pdp, feat_names, avgtaus, sec, H, out_path, ncols=3):
    """Partial-dependence panel: one subplot per feature, q[:,j] vs pdp[:,j], titled
    with the feature name and its avgtau (smooth-vs-hard visible alongside the number).
    q, pdp are (npoints, k) arrays from HTBpartialplot."""
    k = len(feat_names)
    nrows = int(np.ceil(k / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 2.7 * nrows), squeeze=False)
    for j in range(nrows * ncols):
        ax = axes[j // ncols][j % ncols]
        if j >= k:
            ax.axis('off'); continue
        ax.plot(np.asarray(q)[:, j], np.asarray(pdp)[:, j], color=OKABE_ITO[0], lw=1.6)
        at = avgtaus[j]
        at_txt = f'{at:.1f}' if (at is not None and np.isfinite(at)) else 'n/a'
        ax.set_title(f'{feat_names[j]}\n$\\overline{{\\tau}}$={at_txt}', fontsize=9)
        ax.tick_params(labelsize=7)
    fig.suptitle(f'Partial dependence — {sec}, H={H}d (other features held at mean)',
                 fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(out_path, format='pdf'); plt.show(); plt.close(fig)
    return out_path


print('τ^w plotting helpers defined: plot_tau_heatmap, plot_importance_by_taxonomy, '
      'plot_pdp_panel (matplotlib-only, array inputs)')


In [ ]:
# ── Figure 1 (τ^w heatmap) + Figure 2 (importance by taxonomy) + band self-test
# Figure 1 defaults to the MOST DATA-RICH walk-forward fold (expanding window ⇒ the
# last fold trains on the longest history). τ^w from early/narrow folds is noisy, so
# the heatmap deliberately uses the richest fold and names it in the caption.
FIG_TAU_HEATMAP = f'{OUT_DIR}/v5_fig_tau_w_heatmap.pdf'
FIG_TAX_IMP     = f'{OUT_DIR}/v5_fig_importance_taxonomy.pdf'

# Band-helper self-test (tutorial thresholds: ≤7 strong, 7–15 mild, 15–25 weak, >25 none)
_band_checks = {5: 'strong', 12: 'mild', 22: 'weak', 30: 'none'}
for _v, _exp in _band_checks.items():
    assert tau_w_band(_v) == _exp, f'band map wrong: {_v}→{tau_w_band(_v)} (exp {_exp})'
assert tau_w_band(7) == 'strong' and tau_w_band(15) == 'mild' and tau_w_band(25) == 'weak'
assert tau_w_band(float("nan")) == 'na' and tau_w_band(None) == 'na'
print(f'[PASS] tau_w_band thresholds: {_band_checks} (+ boundaries 7/15/25, na)')

# Figure 1 — τ^w heatmap on the richest fold
if 'df_wf_tau' in dir() and len(df_wf_tau):
    _fold_order = [f[0] for f in WF_FOLDS]                     # WF folds, earliest→latest
    _present = [f for f in _fold_order if f in set(df_wf_tau['fold'])]
    TAU_HEATMAP_FOLD = _present[-1] if _present else df_wf_tau['fold'].iloc[-1]
    _sub = df_wf_tau[df_wf_tau['fold'] == TAU_HEATMAP_FOLD]
    plot_tau_heatmap(_sub, TAU_HEATMAP_FOLD, FIG_TAU_HEATMAP)
    print(f'[FIG1] τ^w heatmap (fold={TAU_HEATMAP_FOLD}) → {FIG_TAU_HEATMAP}')
    print(f'  Caption: Importance-weighted smoothness τ^w by NOK tenor × forecast '
          f'horizon, walk-forward fold "{TAU_HEATMAP_FOLD}" (the most data-rich, '
          f'expanding-window fold). Cell shading: strong ≤7, mild 7–15, weak 15–25, '
          f'none >25. Early/narrow folds omitted as τ^w is noisy there.')
else:
    print('[FIG1] skipped — no df_wf_tau (run the WF sweep with want_tau=True first).')

# Figure 2 — importance aggregated into the 8 taxonomy buckets (curve vs macro, §4.5)
# Aggregates the per-feature `df` importance column (from HTBweightedtau, persisted in
# df_tau_perfeat) through the existing bucket_feature; falls back to the HTBrelevance
# importance (wf_imps) only if the τ sidecar is unavailable.
if 'df_tau_perfeat' in dir() and len(df_tau_perfeat):
    _imp_src = [{'security': _s, 'horizon': _h, 'fold': _f,
                 'imp': dict(zip(_g['feature'], _g['importance']))}
                for (_s, _h, _f), _g in
                df_tau_perfeat.groupby(['security', 'horizon', 'fold'])]
elif 'wf_imps' in dir() and len(wf_imps):
    _imp_src = wf_imps
else:
    _imp_src = []
if _imp_src:
    plot_importance_by_taxonomy(_imp_src, FIG_TAX_IMP, per_tenor=False)
    print(f'[FIG2] importance-by-taxonomy → {FIG_TAX_IMP}')
    print('  Caption: Mean feature importance (HTBweightedtau df column) by taxonomy '
          'bucket, pooled across NOK tenors/horizons/folds (walk-forward). '
          'Operationalises the curve-vs-macro comparison (§4.5).')
else:
    print('[FIG2] skipped — no per-feature importance yet (run the WF sweep first).')


In [ ]:
# ── Figure 3 (appendix) — partial-dependence panel + single-fold τ^w VALIDATION
# Self-contained diagnostic fit (reuses the bridge prep helpers; NO modelling change)
# for one representative tenor/horizon, on the most data-rich walk-forward fold. This
# is the single-fold validation the handoff asks for AND the source for Figure 3.
# Requires the juliacall HTBoost kernel; degrades gracefully if a call fails.
PDP_SEC      = 'NOR_10Y'    # draft's running example
PDP_H        = 21           # 1-month horizon
PDP_TOPK     = 9            # 3×3 grid of top-importance features
PDP_FOLD_IDX = -1           # last WF fold = most data-rich (expanding window)
FIG_PDP      = f'{OUT_DIR}/v5_fig_pdp_{PDP_SEC}_H{PDP_H}.pdf'


def _jl_guard(call, what):
    """Run a Julia call under HTBoost logger suppression + try/except. None on failure."""
    jl.seval('import Logging; _htb_saved_logger = Logging.global_logger()')
    jl.seval('Logging.global_logger(Logging.SimpleLogger(devnull, Logging.Error))')
    try:
        return call()
    except Exception as e:
        print(f'    [warn] {what} failed: {repr(e)[:70]}')
        return None
    finally:
        jl.seval('Logging.global_logger(_htb_saved_logger)')


def diag_htbfit(sec, H, fold_idx=PDP_FOLD_IDX):
    """Build (output, data, model-matrix colnames) for one tenor/horizon on one WF
    fold, reusing prepare_x_v5 / _to_julia_dates / _build_param / fit_transform_xm_pca."""
    panel = build_panel_v4(df_raw, [sec], H)
    fc = [c for c in panel.columns if c not in META_COLS]
    fold_name, ts, te_end, regime = WF_FOLDS[fold_idx]
    purge = pd.Timestamp(ts) - pd.tseries.offsets.BDay(H - 1)
    tr = panel[panel['date'] < purge].copy()
    assert len(tr) >= MIN_TRAIN_OBS, f'{sec} H={H} fold={fold_name}: only {len(tr)} train rows'
    x_tr, _, _ = fit_transform_xm_pca(tr[fc], tr[fc])          # PCA fit on train rows only
    x_prep = prepare_x_v5(x_tr)
    x_jl   = jl.DataFrame(x_prep)
    param  = _build_param(FROZEN_CONFIG, H)
    data   = jl.HTBdata(tr['y'].to_numpy(float), x_jl, param, _to_julia_dates(tr['date']))
    output = _jl_guard(lambda: jl.HTBfit(data, param), 'HTBfit (diag)')
    return output, data, list(x_prep.columns), fold_name, len(tr)

try:
    _out, _data, _cols, _fold, _ntr = diag_htbfit(PDP_SEC, PDP_H)
    print(f'=== τ^w VALIDATION — {PDP_SEC} H={PDP_H} fold={_fold} (n_train={_ntr}, '
          f'{len(_cols)} model-matrix features) ===')

    # τ^w extraction (guarded; introspect the tuple fields, do not assume names)
    _wt = _jl_guard(lambda: jl.HTBweightedtau(_out, _data, verbose=False), 'HTBweightedtau (diag)')
    _rel = _jl_guard(lambda: jl.HTBrelevance(_out, _data, verbose=False), 'HTBrelevance (diag)')
    _diag_tau_w, _diag_tau_tab, _diag_imp = None, None, {}
    if _wt is not None:
        _diag_tau_w, _diag_tau_tab = _extract_weightedtau(_wt)   # pinned gavgtau + df table
        print(f'  scalar τ^w (gavgtau = geometric importance-weighted mean) '
              f'= {_diag_tau_w:.3f}  → band={tau_w_band(_diag_tau_w)}')
        print(f'  per-feature avgtau rows = {len(_diag_tau_tab)}  (model-matrix features = {len(_cols)})')
        assert np.isfinite(_diag_tau_w) and 1.0 <= _diag_tau_w <= 40.0, \
            f'τ^w out of plausible 1–40 range: {_diag_tau_w}'
        assert len(_diag_tau_tab) == len(_cols), \
            f'τ table rows ({len(_diag_tau_tab)}) != model features ({len(_cols)})'
        print('  [PASS] τ^w finite, in 1–40, one row per input feature.')
    if _rel is not None:
        _diag_imp = dict(zip([str(n) for n in _rel.fnames], np.asarray(_rel.fi, float)))

    # Figure 3 — PDP panel over the top-k features by importance
    _rank_src = _diag_imp if _diag_imp else (
        dict(zip(_diag_tau_tab['feature'], _diag_tau_tab['importance'])) if _diag_tau_tab is not None else {})
    _topk = [f for f, _ in sorted(_rank_src.items(), key=lambda kv: -kv[1])
             if f in _cols][:PDP_TOPK]
    if _out is not None and _topk:
        _idx1 = [_cols.index(f) + 1 for f in _topk]            # 1-based Julia column indices
        _idx_jl = jl.seval('x -> Int.(collect(x))')(_idx1)
        _pp = _jl_guard(lambda: jl.HTBpartialplot(_data, _out, _idx_jl), 'HTBpartialplot')
        if _pp is not None:
            _q_jl, _pdp_jl = _pp
            _q, _pdp = np.asarray(_q_jl, float), np.asarray(_pdp_jl, float)
            _av = ({r.feature: r.avgtau for r in _diag_tau_tab.itertuples()}
                   if _diag_tau_tab is not None else {})
            _avgs = [_av.get(f, np.nan) for f in _topk]
            plot_pdp_panel(_q, _pdp, _topk, _avgs, PDP_SEC, PDP_H, FIG_PDP, ncols=3)
            print(f'[FIG3] partial-dependence panel → {FIG_PDP}')
            print(f'  Caption: Partial dependence of the top-{PDP_TOPK} features (by '
                  f'importance) for {PDP_SEC} at H={PDP_H}d, walk-forward fold '
                  f'"{_fold}"; all other features held at their mean. Subtitles show '
                  f'each feature’s avgtau (low ⇒ smooth, high ⇒ hard split).')
    else:
        print('[FIG3] skipped — no fit/importance available.')

    # Sanity report: NOR_10Y τ^w across horizons (from the sweep, if it has run)
    if 'df_wf_tau' in dir() and len(df_wf_tau):
        _nor = df_wf_tau[(df_wf_tau['security'] == PDP_SEC)]
        if len(_nor):
            print(f'\n  {PDP_SEC} τ^w across horizons (richest fold per horizon, sweep):')
            for _h in sorted(_nor['horizon'].unique()):
                _hh = _nor[_nor['horizon'] == _h]
                _last = _hh.iloc[-1]
                print(f'    H={_h:<3d}  τ^w={_last["tau_w"]:.2f}  band={_last["tau_w_band"]}')
    print('\n  NOTE: τ^w is only interpretable under modality=:accurate/:compromise in '
          'large samples; early/narrow-coverage folds (post-GFC) are noisy.')
except Exception as _e:
    print(f'[τ^w diagnostic skipped] {repr(_e)[:120]}')
    print('  (requires the juliacall HTBoost kernel — run inside the user’s Julia env.)')


### Pooled-vs-per-security comparison
If the pooled notebook has written a results CSV with the shared schema, merge it and compare the two on the identical metric set by regime. Skips gracefully when the pooled CSV is absent.

In [ ]:
# ── pooled-vs-per-security comparison (graceful if pooled CSV absent) ────
# If a pooled CSV with the SHARED schema is present at POOLED_CSV_PATH, merge and
# render pooled-vs-per-security on the identical metric set. Their agreement /
# disagreement is itself a reported result.
if not os.path.exists(POOLED_CSV_PATH):
    print(f'[comparison skipped] No pooled CSV at {POOLED_CSV_PATH!r}.')
    print('  When the pooled notebook writes its v5-schema long CSV here, this cell merges')
    print('  per-security vs pooled on DirAcc / Campbell–Thompson R² / CW / DM-Harvey / PT,')
    print('  by regime, with MTC survivor counts.')
else:
    pooled = pd.read_csv(POOLED_CSV_PATH)
    _missing = [c for c in SHARED_COLS if c not in pooled.columns]
    if _missing:
        print(f'[comparison warning] pooled CSV missing shared cols: {_missing[:8]}... — '
              f'fix the pooled writer to match the metrics contract.')
    shared = [c for c in SHARED_COLS if c in pooled.columns and c in df_all.columns]
    merged = pd.concat([df_all[shared].assign(src='per_security'),
                        pooled[shared].assign(src='pooled')], ignore_index=True)
    oos = merged[merged['sample'] == 'oos']
    print('=== Pooled vs per-security — OOS metric set ===')
    tab = oos.groupby(['src', 'validation_scheme']).agg(
        n=('dir_acc', 'size'), mean_DA=('dir_acc', 'mean'),
        pct_gt50=('dir_acc', lambda s: float((s > 0.5).mean() * 100)),
        mean_CT_R2=('ct_r2_oos', 'mean'),
        mean_CW_p=('cw_pval', 'mean'), mean_DMH_p=('dmh_pval', 'mean'),
        mean_PT_p=('pt_pval', 'mean'))
    print(tab.to_string(float_format=lambda x: f'{x:.3f}'))
    print('\n  By regime (OOS DirAcc):')
    print(oos.pivot_table(index='regime', columns='src', values='dir_acc', aggfunc='mean')
          .to_string(float_format=lambda x: f'{x:.3f}'))
    merged.to_csv(f'{OUT_DIR}/v5_pooled_comparison.csv', index=False)
    print(f'\n  Saved: {OUT_DIR}/v5_pooled_comparison.csv')
    print('  Reading: signal present in pooled but absent per-security ⇒ borrowed strength '
          '(cross-sectional pooling), not idiosyncratic per-security skill.')

### Verdict
State the decision criteria fixed before running, then report the observed walk-forward train/OOS gap, the multiple-testing survivors, and the block-CV result under its asymmetric interpretation rule.

In [ ]:
# ── VERDICT (v5) — null confirmation + block-CV asymmetric interpretation ────
print('=== v5 VERDICT ===\n')
oos_wf = df_all[(df_all['validation_scheme'] == 'walk_forward') & (df_all['sample'] == 'oos')]
tr_wf  = df_all[(df_all['validation_scheme'] == 'walk_forward') & (df_all['sample'] == 'train')]

mean_oos   = float(oos_wf['dir_acc'].mean()) if len(oos_wf) else float('nan')
mean_train = float(tr_wf['dir_acc'].mean())  if len(tr_wf)  else float('nan')
mean_gap   = mean_train - mean_oos
mean_ctr2  = float(oos_wf['ct_r2_oos'].mean()) if len(oos_wf) else float('nan')

print('Decision criteria (stated before running):')
print('  Null confirmed: OOS DirAcc ~ noise AND no security clears MTC across the grid.')
print('  Genuine lead:   clears Bonferroni/BH-FDR AND positive across MULTIPLE regimes.')
print()
print('Observed (walk-forward, causal):')
print(f'  mean train DirAcc:            {mean_train:.4f}')
print(f'  mean OOS   DirAcc:            {mean_oos:.4f}   (noise ≈ 0.50)')
print(f'  mean overfit gap (train−OOS): {mean_gap:+.4f}   ← the Ch.7-vs-Ch.8 headline')
print(f'  mean Campbell–Thompson R²:    {mean_ctr2:+.4f}   (negative ⇒ worse than RW)')
print(f'  walk-forward MTC survivors:   Bonferroni={b_wf}  BH-FDR={h_wf}  (N={N_wf})')
if len(oos_wf):
    print('\n  OOS DirAcc by regime:')
    for reg, m in oos_wf.groupby('regime')['dir_acc'].mean().items():
        print(f'    {reg:<8s} {m:.4f}')

print('\nVerdict:')
print('─' * 78)
if b_wf == 0 and h_wf == 0:
    print(
        f'v5 confirms the null. Across the full {{horizon × tenor × regime}} grid the per-security '
        f'walk-forward OOS DirAcc averages {mean_oos:.3f} with a large in-sample fit '
        f'({mean_train:.3f}) collapsing to noise OOS (gap {mean_gap:+.3f}); the mean '
        f'Campbell–Thompson R² is {mean_ctr2:+.3f} (no improvement on the random walk). '
        f'No security survives Bonferroni or BH-FDR correction (N={N_wf}). PCA compression of '
        f'the cross-market block and inner-CV-frozen regularisation reduce the overfitting surface '
        f'without changing the conclusion. The directional ML signal does NOT beat the random walk OOS.'
    )
else:
    print(
        f'CAUTION: {b_wf} Bonferroni / {h_wf} BH-FDR survivor(s) in the walk-forward family (N={N_wf}). '
        f'Before reporting, apply the NOR_30Y standard: require positive DirAcc across MULTIPLE regimes '
        f'(GFC, ZIRP, COVID, Hiking), not Hiking-only; re-audit for leakage; check economic plausibility.'
    )
print('─' * 78)

# Block-CV / LORO asymmetric interpretation (robustness, never the headline).
oos_bl = df_all[(df_all['sample'] == 'oos') & (df_all['validation_scheme'].isin(['block_cv', 'loro']))]
if len(oos_bl):
    mean_bl = float(oos_bl['dir_acc'].mean())
    print('\nBlock-CV / LORO robustness (NON-causal — learnability diagnostic, NOT a forecast):')
    print(f'  mean block-CV/LORO OOS DirAcc: {mean_bl:.4f}   (non-causal MTC: '
          f'Bonferroni={b_bl} BH-FDR={h_bl}, N={N_bl})')
    print('  Ex-ante asymmetric rule:')
    if abs(mean_bl - 0.5) < 0.02:
        print('   → ≈ 0.50: STRONGER confirmation of the null — even with future data surrounding the')
        print('     held-out block, no stable feature→change relationship transfers in.')
    elif mean_bl > mean_oos + 0.02:
        print('   → > walk-forward while walk-forward ≈ 0.50: attributable to the FUTURE-INFORMATION')
        print('     advantage / larger training set. It flags NON-STATIONARITY, NOT tradeable skill.')
        print('     It does NOT enter Part C and is NOT the primary OOS finding.')
    else:
        print('   → comparable to walk-forward: consistent with the null.')
    print('  (Bergmeir-Hyndman-Koo 2018; López de Prado 2018 purged K-fold + embargo.)')
print('\n[v5 COMPLETE]')